In [ ]:
"""
T-Mobile Competitive Offer Report — Extraction Pipeline (Notebook 1)
=====================================================================
Bronze (PDF -> classified pages) -> Silver (structured offer/comparison tables)
-> Review queue (anything uncertain or unrecognized).

STRUCTURE
  0. Imports & config
  1. Page-type registry — the single source of truth for how each page type
     is handled (skip / text-only / text-first-grid / image), and whether
     bronze stores an image for it. Locked in here; extending it is how a
     genuinely new page type gets extraction built later.
  2. Shared utilities — manifest/hash helpers, BigQuery write helpers, text
     reconstruction, page classification (incl. self-service pattern
     additions and LLM auto-classification), Gemini call wrappers,
     hallucination guard, carrier normalization, dedup, logging.
  3. Stage 1 — bronze: PDF -> per-page text/image -> classified rows.
  4. Diagnostics — inspect any page/run directly.
  5. Stage 2, text-only tier — ci_headlines, national_retail_readout,
     major_offer_changes, voice_plan_comparison, business_internet_comparison,
     prepaid_price_grid_detail, postpaid_service_offers.
  6. Stage 2, text-first grid tier — postpaid_bts_offers, cable_mvno_offers,
     prepaid_brand_offers.
  7. Stage 2, image tier — postpaid device offers (apple/samsung_google/
     motorola_affordable) and business_device_offers.
  8. Orchestration — one call to run every Stage 2 extractor.

Notebook 2 (not in this file) will read the review queue and build gold.
"""

# ============================================================================
# 0. IMPORTS & CONFIG
# ============================================================================
# %% [Cell 1: imports & config]
import base64
import traceback
import hashlib
import json
import re
import uuid
from datetime import date, datetime

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.cloud import bigquery
from pypdf import PdfReader
import fitz  # PyMuPDF — !pip install pymupdf if not already available

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig, Part

PROJECT_ID = "prj-dbi-prd-1"
DATASET_ID = "ds_dbi_digitalmedia_automation"
GEMINI_MODEL = "gemini-2.5-pro"
GEMINI_LOCATION = "us-central1"  # confirm this is enabled for your project — your BQ
                                  # dataset being in us-west1 says nothing about Vertex
                                  # AI region availability/policy

bq = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=GEMINI_LOCATION)

# ---- Table names, grouped by layer -----------------------------------------
MANIFEST_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_uploadManifest_weekly"
BRONZE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_pdfPages_weekly"
REVIEW_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_review_flaggedOffers_weekly"
PAGE_TYPE_PATTERNS_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_control_pageTypePatterns"
# ^ no _weekly suffix — a slowly-growing reference table, not a weekly snapshot.

SILVER_CI_HEADLINES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_ciHeadlines_weekly"
SILVER_RETAIL_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_nationalRetailReadout_weekly"
SILVER_MAJOR_CHANGES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_majorOfferChanges_weekly"
SILVER_VOICE_PLAN_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_voicePlanComparison_weekly"
SILVER_BIZ_INTERNET_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_businessInternetComparison_weekly"
SILVER_PREPAID_GRID_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_prepaidPriceGridDetail_weekly"
SILVER_SERVICE_OFFERS_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_postpaidServiceOffers_weekly"
SILVER_GRID_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_carrierOfferGrid_weekly"
SILVER_DEVICE_OFFERS_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_silver_postpaidDeviceOffers_weekly"

# Gold: same schema as its silver counterpart, populated with only the
# gold-eligible subset (see is_gold_eligible / promote_to_gold below).
# Notebook 2 will add rows back here as review items get resolved.
GOLD_CI_HEADLINES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_ciHeadlines_weekly"
GOLD_RETAIL_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_nationalRetailReadout_weekly"
GOLD_MAJOR_CHANGES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_majorOfferChanges_weekly"
GOLD_VOICE_PLAN_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_voicePlanComparison_weekly"
GOLD_BIZ_INTERNET_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_businessInternetComparison_weekly"
GOLD_PREPAID_GRID_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_prepaidPriceGridDetail_weekly"
GOLD_SERVICE_OFFERS_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_postpaidServiceOffers_weekly"
GOLD_GRID_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_carrierOfferGrid_weekly"
GOLD_DEVICE_OFFERS_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_gold_postpaidDeviceOffers_weekly"


# ============================================================================
# 1. PAGE-TYPE REGISTRY — locked in here. Two independent properties per type:
#      store_image      — does Stage 1 render+store a PNG for this page?
#      stage2_strategy  — which Stage 2 pipeline handles it:
#                          "skip" | "text_only" | "text_first_grid" | "image"
#    store_image is True for every table/grid-shaped page even when the
#    current strategy is text-only — the image is there and ready if a
#    future run shows this page type needs escalation, with no Stage 1
#    re-run required to get it.
#    A page type NOT in this registry gets no image and no Stage 2 extraction
#    (falls through as effectively "unhandled") — this is what happens to a
#    genuinely new page type until it's deliberately added below.
# ============================================================================

# %% [Cell 2: page-type registry]
PAGE_TYPE_REGISTRY = {
    # -- always skipped: no offer content -----------------------------------
    "title_slide":                             {"store_image": False, "stage2_strategy": "skip"},
    "confidentiality_notice":                  {"store_image": False, "stage2_strategy": "skip"},
    "toc":                                     {"store_image": False, "stage2_strategy": "skip"},
    "section_divider":                         {"store_image": False, "stage2_strategy": "skip"},
    "closing_slide":                           {"store_image": False, "stage2_strategy": "skip"},

    # -- text-only: narrative/bulleted content, no image ever helped --------
    "ci_headlines":                            {"store_image": False, "stage2_strategy": "text_only"},
    "national_retail_readout":                 {"store_image": False, "stage2_strategy": "text_only"},

    # -- text-only: clean structured text, image stored for future escalation
    "major_offer_changes":                     {"store_image": True,  "stage2_strategy": "text_only"},
    "voice_plan_comparison":                   {"store_image": True,  "stage2_strategy": "image"},
    "business_internet_comparison":            {"store_image": True,  "stage2_strategy": "image"},
    "prepaid_price_grid_detail":                {"store_image": True,  "stage2_strategy": "text_only"},
    "postpaid_service_offers":                  {"store_image": True,  "stage2_strategy": "text_only"},

    # -- text-first grid: carrier x item x offer shape, text proved adequate
    "postpaid_bts_offers":                      {"store_image": True,  "stage2_strategy": "text_first_grid"},
    "cable_mvno_offers":                        {"store_image": True,  "stage2_strategy": "text_first_grid"},
    "prepaid_brand_offers":                     {"store_image": True,  "stage2_strategy": "text_first_grid"},

    # -- image: same carrier x item x offer shape, but text alone proved
    #    unreliable (cross-column/cross-tier attribution errors)
    "postpaid_device_offers_apple":             {"store_image": True,  "stage2_strategy": "image"},
    "postpaid_device_offers_samsung_google":    {"store_image": True,  "stage2_strategy": "image"},
    "postpaid_device_offers_motorola_affordable": {"store_image": True, "stage2_strategy": "image"},
    "business_device_offers":                   {"store_image": True,  "stage2_strategy": "image"},
}

NON_CONTENT_PAGE_TYPES = {pt for pt, cfg in PAGE_TYPE_REGISTRY.items() if cfg["stage2_strategy"] == "skip"}


# ============================================================================
# 2. SHARED UTILITIES
# ============================================================================

# %% [Cell 3: shared utils — manifest/hash/date helpers]
# ---- 2a. Manifest / hashing / date helpers ---------------------------------

def parse_date_from_filename(filename: str):
    """MCI's convention: '..._MMDDYY.pdf' e.g. Competitive_Offer_Report_050826.pdf"""
    m = re.search(r"_(\d{2})(\d{2})(\d{2})\.pdf$", filename, re.IGNORECASE)
    if not m:
        return None
    mm, dd, yy = m.groups()
    return datetime.strptime(f"20{yy}-{mm}-{dd}", "%Y-%m-%d").date()


def verify_date_from_content(pdf_bytes: bytes, expected_date):
    """Cross-check the filename-derived date against text actually inside the PDF
    (CI Headlines page reliably contains 'COMPETITIVE INTELLIGENCE HEADLINES <Month D, YYYY>').
    Cheap: only reads a couple of pages, not the whole deck."""
    from io import BytesIO
    reader = PdfReader(BytesIO(pdf_bytes))
    for page in reader.pages[:6]:
        text = page.extract_text() or ""
        m = re.search(r"([A-Z][a-z]+ \d{1,2}, 20\d{2})", text)
        if m:
            found = datetime.strptime(m.group(1), "%B %d, %Y").date()
            return found == expected_date, found
    return None, None  # couldn't verify — not necessarily a problem, just flag as unverified


def file_hash(pdf_bytes: bytes) -> str:
    return hashlib.sha256(pdf_bytes).hexdigest()


def get_active_manifest_row(report_date):
    query = f"""
        SELECT run_id, file_name, file_hash, uploaded_at
        FROM `{MANIFEST_TABLE}`
        WHERE report_date = @report_date AND status = 'active'
        ORDER BY uploaded_at DESC
        LIMIT 1
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
        ),
    )
    rows = list(job.result())
    return rows[0] if rows else None


def get_latest_run_id(report_date):
    """Look up the current active run_id for a report_date directly, instead of
    copy-pasting one out of earlier log output."""
    row = get_active_manifest_row(report_date)
    if row is None:
        print(f"No active manifest row for {report_date}. Check show_bronze_state() or upload the file first.")
        return None
    print(f"Using run_id {row['run_id']} (uploaded {row['uploaded_at']}, file {row['file_name']!r})")
    return row["run_id"]


# %% [Cell 4: shared utils — BigQuery write helpers]
# ---- 2b. BigQuery write helpers --------------------------------------------

def write_row(table_id, row: dict):
    """Load-job insert instead of streaming (insert_rows_json) — load jobs write
    directly to the table with no streaming buffer, so a DELETE shortly after
    (e.g. while testing) won't hit BigQuery's 'can't DML rows in streaming buffer'
    error."""
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    job = bq.load_table_from_json([row], table_id, job_config=job_config)
    job.result()


def write_rows(table_id, rows: list):
    if not rows:
        return
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    job = bq.load_table_from_json(rows, table_id, job_config=job_config)
    job.result()


def write_manifest_row(report_date, run_id, file_name, hash_, status):
    write_row(MANIFEST_TABLE, {
        "report_date": report_date.isoformat(), "run_id": run_id, "file_name": file_name,
        "file_hash": hash_, "status": status, "uploaded_at": datetime.utcnow().isoformat(),
        "uploaded_by": "notebook1",
    })


def delete_existing_run(report_date) -> bool:
    """Delete-then-replace for a report_date's bronze + manifest rows. Returns
    False if it couldn't complete, so the caller aborts rather than writing new
    data on top of old data that failed to clear."""
    for table_id in (BRONZE_TABLE, MANIFEST_TABLE):
        try:
            bq.query(
                f"DELETE FROM `{table_id}` WHERE report_date = @report_date",
                job_config=bigquery.QueryJobConfig(
                    query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
                ),
            ).result()
        except Exception as e:
            if "streaming buffer" in str(e):
                print(f"Can't delete from {table_id} yet — those rows are still in BigQuery's "
                      f"streaming buffer (usually clears within a couple hours on its own). "
                      f"If this is test data, run reset_tables() and re-upload.")
                return False
            raise
    return True


def delete_silver_rows_for_date(table_id, report_date):
    """Every silver table gets delete-before-write: a fresh extraction for a
    report_date should fully replace whatever a prior run_id wrote for that
    same date, not append alongside it."""
    bq.query(
        f"DELETE FROM `{table_id}` WHERE report_date = @report_date",
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
        ),
    ).result()


def dedupe_review_rows(rows: list) -> list:
    """Collapses review flags that share the same (source_page, carrier,
    offer_amount, flag_reason) into one representative row. Confirmed real
    cause of inflated review volume: one offer applying to several devices
    (device_group/item_group fan-out) gets emitted as N separate silver rows
    by design — correct for the data — but when the shared offer_amount fails
    the guard, that's N near-identical flags for what a reviewer would
    recognize as one question, not N different ones. One flag noting how many
    items it affects is just as actionable and far less exhausting to review."""
    grouped = {}
    order = []
    for r in rows:
        key = (r.get("source_page"), r.get("carrier"), r.get("offer_amount"), r.get("flag_reason"))
        if key not in grouped:
            grouped[key] = {**r, "_affected_items": [r.get("item_name")]}
            order.append(key)
        else:
            grouped[key]["_affected_items"].append(r.get("item_name"))
    out = []
    for key in order:
        g = grouped[key]
        items = g.pop("_affected_items")
        if len(items) > 1:
            shown = [str(i) for i in items[:8] if i]
            more = f", +{len(items) - len(shown)} more" if len(items) > len(shown) else ""
            g["item_name"] = f"{items[0]} (+{len(items) - 1} more item(s) with this same offer)"
            g["detail"] = f"{g.get('detail') or ''} — affects {len(items)} items: {', '.join(shown)}{more}".strip(" —")
        out.append(g)
    return out


def write_review_rows(rows: list):
    if not rows:
        return
    deduped = dedupe_review_rows(rows)
    if len(deduped) < len(rows):
        print(f"  (collapsed {len(rows)} flagged row(s) into {len(deduped)} distinct issue(s) for review — "
              f"same offer applying to multiple items no longer multiplies review volume)")
    write_rows(REVIEW_TABLE, deduped)


def reset_tables():
    """NOT run automatically — a reset button for test data, not something to
    run against a table you actually care about."""
    bq.query(f"DROP TABLE IF EXISTS `{BRONZE_TABLE}`").result()
    bq.query(f"DROP TABLE IF EXISTS `{MANIFEST_TABLE}`").result()
    print("Dropped both tables. Re-run table creation below, then re-upload your files.")


def fetch_bronze_pages(run_id, page_types):
    query = f"""
        SELECT page_number, page_type, text_extraction_raw
        FROM `{BRONZE_TABLE}`
        WHERE run_id = @run_id AND page_type IN UNNEST(@page_types)
        ORDER BY page_number
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
            bigquery.ArrayQueryParameter("page_types", "STRING", page_types),
        ]),
    )
    return list(job.result())


def fetch_bronze_pages_with_image(run_id, page_types):
    query = f"""
        SELECT page_number, page_type, text_extraction_raw, page_image_png
        FROM `{BRONZE_TABLE}`
        WHERE run_id = @run_id AND page_type IN UNNEST(@page_types)
        ORDER BY page_number
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
            bigquery.ArrayQueryParameter("page_types", "STRING", page_types),
        ]),
    )
    return list(job.result())


# %% [Cell 5: shared utils — text reconstruction & image rendering]
# ---- 2c. Per-page text reconstruction & image rendering --------------------

def reconstruct_layout_text(page, y_tolerance=3):
    """Cluster words into horizontal rows by y-position, sort left-to-right
    within each row. Used directly for narrative pages, and as the input to
    classify_page (classification always uses this version, regardless of
    which version ultimately gets stored — see run_extraction_pipeline)."""
    words = page.get_text("words")  # (x0, y0, x1, y1, word, block, line, word_no)
    if not words:
        return ""
    words = sorted(words, key=lambda w: (round(w[1] / y_tolerance), w[0]))
    lines, current_bucket, current_line = [], None, []
    for w in words:
        bucket = round(w[1] / y_tolerance)
        if current_bucket is None or bucket == current_bucket:
            current_line.append(w)
            current_bucket = bucket
        else:
            lines.append(" ".join(x[4] for x in sorted(current_line, key=lambda x: x[0])))
            current_line, current_bucket = [w], bucket
    if current_line:
        lines.append(" ".join(x[4] for x in sorted(current_line, key=lambda x: x[0])))
    return "\n".join(lines)


KNOWN_COLUMN_HEADERS = {
    "t-mobile", "verizon", "at&t", "att", "xfinity", "spectrum", "comcast",
    "optimum", "metro", "boost", "cricket", "smartphones",
}


def find_header_columns(page, y_tolerance=3, min_words_in_row=3, min_gap=15):
    """Look for the column-header row in the top third of the page. Two
    signals combined: (1) skip the very first row-cluster (the page title,
    not the header row); (2) among what's left, prefer rows containing known
    carrier vocabulary over rows that just have the most spaced-out words."""
    words = page.get_text("words")
    if not words:
        return None
    page_height = page.rect.height
    candidates = [w for w in words if w[1] < page_height * 0.35]
    candidates = sorted(candidates, key=lambda w: (round(w[1] / y_tolerance), w[0]))
    rows = {}
    for w in candidates:
        bucket = round(w[1] / y_tolerance)
        rows.setdefault(bucket, []).append(w)

    sorted_buckets = sorted(rows.keys())
    if len(sorted_buckets) > 1:
        rows = {b: rows[b] for b in sorted_buckets[1:]}  # drop the title row

    scored = []
    for row_words in rows.values():
        row_words = sorted(row_words, key=lambda w: w[0])
        if len(row_words) < min_words_in_row:
            continue
        gaps = [row_words[i + 1][0] - row_words[i][2] for i in range(len(row_words) - 1)]
        if not gaps or min(gaps) <= min_gap:
            continue
        vocab_hits = sum(1 for w in row_words if w[4].lower().strip(":") in KNOWN_COLUMN_HEADERS)
        scored.append((vocab_hits, len(row_words), row_words))

    if not scored:
        return None
    scored.sort(key=lambda t: (t[0], t[1]), reverse=True)
    return [w[0] for w in scored[0][2]]


def reconstruct_columns(page, y_tolerance=3):
    """Column-aware alternative to reconstruct_layout_text for grid-table
    pages. Detects column boundaries from a header row, buckets every word
    into its column by x-position FIRST, then row-clusters within each column
    separately — avoids cross-column bleeding when different carriers wrap to
    different numbers of lines for the same conceptual row."""
    boundaries = find_header_columns(page, y_tolerance=y_tolerance)
    if not boundaries:
        return reconstruct_layout_text(page, y_tolerance=y_tolerance)  # fallback

    words = page.get_text("words")
    columns = {i: [] for i in range(len(boundaries))}
    for w in words:
        col_idx = 0
        for i, b in enumerate(boundaries):
            if w[0] >= b:
                col_idx = i
        columns[col_idx].append(w)

    blocks = []
    for i in range(len(boundaries)):
        col_words = sorted(columns[i], key=lambda w: (round(w[1] / y_tolerance), w[0]))
        lines, bucket, line = [], None, []
        for w in col_words:
            b = round(w[1] / y_tolerance)
            if bucket is None or b == bucket:
                line.append(w)
                bucket = b
            else:
                lines.append(" ".join(x[4] for x in sorted(line, key=lambda x: x[0])))
                line, bucket = [w], b
        if line:
            lines.append(" ".join(x[4] for x in sorted(line, key=lambda x: x[0])))
        blocks.append(f"[COLUMN {i + 1}]\n" + "\n".join(lines))
    return "\n\n".join(blocks)


def render_page_image(page, dpi=150) -> bytes:
    pix = page.get_pixmap(dpi=dpi)
    return pix.tobytes("png")


# %% [Cell 6: shared utils — page classification]
# ---- 2d. Page classification (built-in patterns + self-service additions +
#          LLM-assisted auto-classification for anything still unrecognized) -

PAGE_TYPE_PATTERNS = [
    (r"Table of Contents", "toc"),
    (r"NOTICE OF CONFIDENTIALITY", "confidentiality_notice"),
    (r"COMPETITIVE INTELLIGENCE HEADLINES", "ci_headlines"),
    (r"CI Headlines", "section_divider"),  # short divider title, distinct from the phrase above
    (r"Major Offer Changes Effective", "major_offer_changes"),
    (r"Post-Paid Apple Smartphone Offers", "postpaid_device_offers_apple"),
    (r"Post-Paid Samsung and Google Smartphone Offers", "postpaid_device_offers_samsung_google"),
    (r"Post-Paid Motorola, Affordable Phones and BYOD", "postpaid_device_offers_motorola_affordable"),
    (r"Post-Paid BTS \(Watches\)", "postpaid_bts_offers"),
    (r"Post-Paid BTS \(Tablet\)", "postpaid_bts_offers"),
    (r"Post-Paid BTS & Other Offers", "postpaid_bts_offers"),
    (r"Post-Paid Service Offers", "postpaid_service_offers"),
    (r"Postpaid Cable MVNO Handset Offers", "cable_mvno_offers"),
    (r"Cable MVNO Service Pricing", "cable_mvno_offers"),
    (r"Business\s*[-–]\s*Flagship Device Offers", "business_device_offers"),
    (r"Key Highlights\s*[-–]\s*Week Ending", "national_retail_readout"),  # more reliable than the
    (r"Promotions Marketplace", "national_retail_readout"),               # Circana wordmark, which
                                                                            # may render as an image
    (r"Prepaid Brands\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"Flagship Brands/Prepaid\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"Walmart\s*[-–]\s*Current Offers", "prepaid_brand_offers"),
    (r"METRO BY T-MOBILE", "prepaid_price_grid_detail"),
    (r"Competitive Offers\s*&\s*Promotional\s*Updates", "section_divider"),
    (r"Competitive Offer Updates", "section_divider"),  # singular, no "& Promotional" — different divider
    (r"Business Internet Competitive Comparison", "business_internet_comparison"),
    (r"Competitive Comparison:\s*(TFB\s*)?Voice Plan", "voice_plan_comparison"),
    (r"Competitive Offer Report", "title_slide"),
    (r"ARE YOU WITH US", "closing_slide"),
]

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{PAGE_TYPE_PATTERNS_TABLE}` (
        pattern STRING,
        page_type STRING,
        added_by STRING,
        added_at TIMESTAMP,
        notes STRING
    )
""").result()

_custom_patterns_cache = None


def load_custom_patterns(force_refresh=False):
    """Patterns added via add_page_type_pattern (or auto-created below), on top
    of the built-in list above — this is what makes recognizing a new page
    type self-service rather than a code change. Cached per Stage 1 run."""
    global _custom_patterns_cache
    if _custom_patterns_cache is not None and not force_refresh:
        return _custom_patterns_cache
    try:
        rows = bq.query(f"SELECT pattern, page_type FROM `{PAGE_TYPE_PATTERNS_TABLE}` ORDER BY added_at").result()
        _custom_patterns_cache = [(r["pattern"], r["page_type"]) for r in rows]
    except Exception:
        _custom_patterns_cache = []
    return _custom_patterns_cache


def add_page_type_pattern(pattern: str, page_type: str, notes: str = ""):
    """Teach the system a new page type from a notebook cell, without a code
    change. Note: this only makes classify_page() recognize the page — it
    does NOT add extraction for it. Adding a genuinely new type to
    PAGE_TYPE_REGISTRY (and building its extraction function) is still a
    separate, deliberate decision."""
    write_row(PAGE_TYPE_PATTERNS_TABLE, {
        "pattern": pattern, "page_type": page_type, "added_by": "khalid",
        "added_at": datetime.utcnow().isoformat(), "notes": notes,
    })
    load_custom_patterns(force_refresh=True)
    print(f"Added: {pattern!r} -> {page_type!r}. Applies starting with the next Stage 1 run.")


def classify_page(text: str):
    normalized = re.sub(r"\s+", " ", text)  # collapse newlines/multi-space so line-wrap
                                              # differences between slide instances don't matter
    for pattern, page_type in PAGE_TYPE_PATTERNS + load_custom_patterns():
        if re.search(pattern, normalized, re.IGNORECASE):
            return page_type, pattern
    return "unclassified", None


def get_known_page_types():
    """Union of every page_type currently recognized, built dynamically so it
    stays in sync as patterns get added."""
    types = ({pt for _, pt in PAGE_TYPE_PATTERNS} | {pt for _, pt in load_custom_patterns()}
             | set(PAGE_TYPE_REGISTRY.keys()))
    return sorted(types)


def suggest_page_type(page_text: str):
    """LLM-assisted starting point for categorizing an unclassified page —
    prints a suggestion, applies nothing automatically."""
    model = GenerativeModel(GEMINI_MODEL)
    prompt = f"""This is text from one page of a weekly telecom competitive
intelligence report. Suggest:
1. A short snake_case page_type label (e.g. "business_internet_comparison")
2. A one-sentence description of what the page contains
3. Which shape it resembles: a carrier-offer grid (carrier x device/item x offer),
   a narrative/bulleted summary, a diff/changes table (introduced vs removed),
   or a numeric matrix with no carrier dimension — this determines what
   extraction approach would fit.

PAGE TEXT:
{page_text[:3000]}"""
    print(model.generate_content(prompt).text)


PAGE_TYPE_CLASSIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "page_type": {"type": "string"},
        "is_new": {"type": "boolean"},
        "distinctive_phrase": {"type": "string"},
        "reasoning": {"type": "string"},
    },
    "required": ["page_type", "is_new", "distinctive_phrase"],
}

AUTO_CLASSIFY_PROMPT = """You are classifying a page from a recurring weekly telecom \
competitive intelligence report. Page types already handled by this pipeline: \
{known_types}

Look at the page text below and decide:
- If it matches one of the known types above (just worded differently than whatever \
literal phrase currently catches that type), return that exact page_type and \
is_new=false.
- If it's genuinely a new kind of content not covered by any known type, propose a \
short new snake_case page_type name and set is_new=true.

Either way, also return distinctive_phrase: a short literal phrase (5-10 words) near \
the top of this page that would reliably identify this same page type in future \
weekly reports — this becomes a regex pattern, so pick a section title or fixed \
label, not a date or number that will change week to week.

PAGE TEXT:
{page_text}"""


def auto_classify_unclassified_pages(run_id, report_date):
    """LLM classification pass over anything Stage 1's regex patterns missed.
    Matching an existing type is applied immediately (low risk — a routing
    label, not a data value). A genuinely new type is auto-registered too (not
    stuck in limbo), but stays visible via a review flag rather than
    disappearing silently. Note: an auto-created page_type is NOT added to
    PAGE_TYPE_REGISTRY automatically — it gets recognized on future runs, but
    building actual extraction for it is still a deliberate step."""
    pages = fetch_bronze_pages(run_id, ["unclassified"])
    if not pages:
        return
    print(f"  Auto-classifying {len(pages)} unclassified page(s)...")
    known_types = get_known_page_types()
    still_unclassified = []

    for page in pages:
        try:
            prompt = AUTO_CLASSIFY_PROMPT.format(
                known_types=", ".join(known_types), page_text=page["text_extraction_raw"][:3000])
            result = call_gemini_json(prompt, PAGE_TYPE_CLASSIFY_SCHEMA)
            page_type, phrase, is_new = result["page_type"], result["distinctive_phrase"], result["is_new"]

            bq.query(
                f"UPDATE `{BRONZE_TABLE}` SET page_type = @page_type WHERE run_id = @run_id AND page_number = @page_number",
                job_config=bigquery.QueryJobConfig(query_parameters=[
                    bigquery.ScalarQueryParameter("page_type", "STRING", page_type),
                    bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
                    bigquery.ScalarQueryParameter("page_number", "INT64", page["page_number"]),
                ]),
            ).result()
            add_page_type_pattern(re.escape(phrase), page_type,
                                   notes=f"auto-added, run {run_id}, page {page['page_number']}")
            print(f"    Page {page['page_number']}: {'NEW category' if is_new else 'matched existing'} "
                  f"-> page_type={page_type!r}  (pattern: {phrase!r})")

            if is_new:
                write_review_rows([{
                    "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                    "source_table": "bronze", "flag_reason": "new_page_type_auto_created",
                    "carrier": None, "item_name": None, "offer_amount": None,
                    "detail": f"Auto-created page_type '{page_type}' — not yet in PAGE_TYPE_REGISTRY, "
                              f"so no extraction runs for it. Reasoning: {result.get('reasoning', '')}",
                    "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
                }])
        except Exception as e:
            print(f"    Page {page['page_number']}: classification FAILED ({e}) — falling back to plain flag")
            still_unclassified.append(page)

    if still_unclassified:
        write_review_rows([{
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": p["page_number"],
            "source_table": "bronze", "flag_reason": "unclassified_page", "carrier": None,
            "item_name": None, "offer_amount": None,
            "detail": f"No page_type pattern matched, and auto-classification failed: {p['text_extraction_raw'][:150]!r}",
            "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
        } for p in still_unclassified])
        print(f"  Flagged {len(still_unclassified)} page(s) that auto-classification couldn't handle.")


# %% [Cell 7: shared utils — Gemini wrappers, guard, carrier norm, dedup, logging]
# ---- 2e. Gemini call wrappers ----------------------------------------------

def call_gemini_json(prompt: str, schema: dict):
    model = GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(
            response_mime_type="application/json", response_schema=schema, temperature=0,
            max_output_tokens=32768,  # raised after a dense multi-tier fan-out page truncated
                                       # mid-response with the default limit
        ),
    )
    return parse_json_with_recovery(response.text)


def call_gemini_json_with_image(prompt: str, schema: dict, image_bytes: bytes):
    model = GenerativeModel(GEMINI_MODEL)
    image_part = Part.from_data(data=image_bytes, mime_type="image/png")
    response = model.generate_content(
        [image_part, prompt],
        generation_config=GenerationConfig(
            response_mime_type="application/json", response_schema=schema, temperature=0,
            max_output_tokens=32768,
        ),
    )
    return parse_json_with_recovery(response.text)


def parse_json_with_recovery(text: str) -> dict:
    """A truncated response (hit the token limit mid-generation) should lose
    only the records after the cutoff, not the entire page — trim back to the
    last complete array entry and close it out rather than raising. Reports
    the recovered count generically (whatever list is in the result), not
    hardcoded to one schema's key name — every schema here uses a different
    top-level key ("offers", "changes", "rows", "headlines"...)."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        last_complete = text.rfind("},")
        if last_complete == -1:
            raise
        salvaged = text[:last_complete + 1] + "]}"
        result = json.loads(salvaged)  # let this raise if still broken — don't mask a real failure
        recovered = next((len(v) for v in result.values() if isinstance(v, list)), 0)
        print(f"    (response was truncated — recovered {recovered} record(s) before the cutoff)")
        return result


# ---- 2f. Hallucination guard ------------------------------------------------

def extract_numeric_value(text: str):
    """Pull a normalized number out of a dollar-figure string, handling 'k'
    abbreviations (1.1k -> 1100) and commas, so '$1,100', '$1.1k', and '1100'
    all compare equal."""
    if not text:
        return None
    m = re.search(r"([\d,]+\.?\d*)\s*(k)?", str(text), re.IGNORECASE)
    if not m:
        return None
    try:
        num = float(m.group(1).replace(",", ""))
    except ValueError:
        return None
    return num * 1000 if m.group(2) else num


def values_present_in_text(record: dict, keys: list, source_text: str) -> bool:
    """Checks that key extracted values are actually grounded in the bronze
    text for this page. Layered, in order of strictness:
    1. Literal substring match (whitespace-normalized).
    2. Strikethrough collapse: source often shows 'OLD NEW' price pairs back
       to back (an old price struck through, replaced by the current one
       right next to it) — recurring across weeks/page types. The extracted
       value is correctly just the current price, which won't appear as a
       contiguous substring with the old one sitting in between.
    3. Numeric fallback: if the literal string still isn't found, check
       whether ANY dollar figure in the source matches numerically once 'k'
       abbreviations and a leading '<' are normalized away (source '<$1.1k'
       vs extracted '$1100', or source '$50' vs extracted '$50/mo'). Looser
       by design — trades some precision for not flagging correct, usefully-
       normalized extractions as hallucinations."""
    normalized_source = re.sub(r"\s+", " ", source_text)
    collapsed_source = re.sub(r"\$[\d,]+\.?\d*\s+(?=\$)", "", normalized_source)
    source_numbers = {extract_numeric_value(m) for m in
                       re.findall(r"[<>]?\$[\d,]+\.?\d*k?", normalized_source, re.IGNORECASE)}

    for key in keys:
        val = str(record.get(key, "")).strip()
        if not val:
            continue
        if re.sub(r"\s+", " ", val) in collapsed_source:
            continue
        target_num = extract_numeric_value(val)
        if target_num is not None and target_num in source_numbers:
            continue
        return False
    return True


def numeric_present_in_text(value, source_text: str) -> bool:
    """Numeric-only counterpart to values_present_in_text, for fields that are
    genuine numbers (voice plan pricing) rather than formatted strings.
    Matches bare numbers too, not just $-prefixed ones: this table's extreme
    cell density (100+ numeric cells per page) is a strong candidate for
    reconstruct_columns occasionally splitting a '$' from its digits into
    different column buckets, which would explain a 100% guard failure rate
    that's otherwise inexplicable. The extracted price itself has no $ sign
    either (schema type is a plain number), so matching bare digits is really
    the more natural check for this specific table, not just a looser one."""
    if value is None:
        return True
    normalized_source = re.sub(r"\s+", " ", source_text)
    source_numbers = {extract_numeric_value(m) for m in
                       re.findall(r"[<>]?\$?[\d,]+\.?\d*k?", normalized_source, re.IGNORECASE)}
    return float(value) in source_numbers


# ---- 2g. Carrier list & normalization ---------------------------------------

# Hardcoded rather than a live view — carriers are a small, stable set, same
# handful across every page type seen. Extend by hand on the rare occasion a
# genuinely new carrier shows up (the auto-classifier's new_carrier flag will
# surface it in review).
KNOWN_CARRIERS = [
    "T-Mobile", "Verizon", "AT&T", "Xfinity", "Spectrum", "Comcast",
    "Optimum", "Metro by T-Mobile", "Boost Mobile", "Cricket Wireless",
    "Straight Talk", "Total Wireless", "T-Mobile Prepaid", "AT&T Prepaid",
    "Verizon Prepaid",
]


def get_known_carriers():
    return KNOWN_CARRIERS


def normalize_carrier(carrier, known_carriers=None):
    """Collapses naming variants the model sometimes copies literally from a
    page's own column header (e.g. 'Optimum Mobile', 'Xfinity Mobile') back to
    the grounded canonical name, when the base name is already known. This was
    a real, confirmed issue: cable MVNO pages flooded new_carrier flags this
    way — not genuinely new carriers, just header text copied verbatim instead
    of the canonical name the prompt asked for."""
    known_carriers = known_carriers or KNOWN_CARRIERS
    if not carrier:
        return carrier
    if carrier in known_carriers:
        return carrier
    stripped = re.sub(r"\s+Mobile$", "", carrier, flags=re.IGNORECASE).strip()
    return stripped if stripped in known_carriers else carrier


# offer_type normalization — built from an actual distinct-value pull against
# real silver data (SELECT offer_type, COUNT(*) ... GROUP BY offer_type), not
# a guess. Only collapses clusters that distribution genuinely showed were the
# same concept under different spelling/casing (discount, trade-in credit, EIP
# buyout, etc.) — deliberately does NOT force the long tail of legitimately
# distinct, low-volume categories (Loyalty Program, Military Offer, Rebate,
# Referral Bonus...) into a fixed bucket list, since those looked like real
# distinctions, not noise, when checked against the data.
OFFER_TYPE_SYNONYMS = {
    "discount": ["discount", "device discount", "discounted price"],
    "trade-in credit": ["trade-in", "trade-in credit", "added trade-in value", "trade-in offer", "trade offer"],
    "bill credit": ["bill credit", "credit", "discount + bill credits"],
    "eip buyout": ["eip buyout", "eip payoff", "bill payoff"],
    "bogo": ["bogo"],
    "free device": ["free", "device on us", "free device", "free item", "free line"],
    "gift card": ["gift card", "reward card"],
    "bundle discount": ["bundle discount", "bundle", "bundle offer"],
    "multi-line discount": ["multi-line discount", "family plan"],
    "port-in credit": ["port-in", "port-in credit"],
    "plan pricing": ["plan pricing", "plan offer", "plan price"],
}
_OFFER_TYPE_LOOKUP = {variant: canonical for canonical, variants in OFFER_TYPE_SYNONYMS.items() for variant in variants}


def normalize_offer_type(offer_type):
    if not offer_type:
        return offer_type
    return _OFFER_TYPE_LOOKUP.get(offer_type.strip().lower(), offer_type)


# ---- Gold promotion --------------------------------------------------------
# A row is gold-worthy if it passed the value-match guard AND (if it has a
# carrier field at all) its carrier is a recognized one — the same two
# conditions that would otherwise send it to review, computed directly from
# the row itself. This avoids needing to cross-reference against review_rows,
# which have inconsistent key shapes across the very different tables below.

def is_gold_eligible(row: dict, known_carriers=None) -> bool:
    known_carriers = known_carriers or KNOWN_CARRIERS
    if not row.get("values_agree", True):
        return False
    carrier = row.get("carrier")
    if carrier and carrier not in known_carriers:
        return False
    return True


def promote_to_gold(gold_table_id, rows: list, report_date, known_carriers=None) -> list:
    """Writes only the gold-eligible subset to the given gold table (same
    schema as its source silver table). Delete-before-write, same discipline
    as every other write in this pipeline. Returns the gold rows written, so
    the caller can report a count."""
    gold_rows = [r for r in rows if is_gold_eligible(r, known_carriers)]
    if gold_rows:
        delete_silver_rows_for_date(gold_table_id, report_date)
        write_rows(gold_table_id, gold_rows)
    return gold_rows


def warn_if_review_queue_too_large(all_review_rows: list, threshold: int = 100):
    """A review queue in the hundreds isn't a backlog to work through, it's a
    sign the guard itself needs adjustment for whatever table is dominating
    it — surface that loudly rather than let it accumulate silently. Prints
    which flag_reason / which likely table is driving the count, since that's
    the actionable part."""
    if len(all_review_rows) <= threshold:
        return
    print(f"\n{'!' * 80}")
    print(f"! WARNING: {len(all_review_rows)} rows flagged for review this run — over the "
          f"{threshold} you said is the practical ceiling for manual review.")
    print("! This almost always means a guard or extraction issue for one specific table, "
          f"not {len(all_review_rows)} genuine content problems. Breakdown by source_table:")
    counts = {}
    for r in all_review_rows:
        counts[r.get("source_table", "unknown")] = counts.get(r.get("source_table", "unknown"), 0) + 1
    for table, n in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"!   {table}: {n}")
    print(f"{'!' * 80}\n")


# ---- 2h. Dedup ---------------------------------------------------------------

def dedupe_generic(items: list) -> list:
    """Exact-duplicate removal for records within one model response — dense
    pages have shown the same block of records repeated 2-3x in a single
    response, likely a generation-length artifact on long structured outputs."""
    seen, out = set(), []
    for it in items:
        key = tuple(sorted(it.items()))
        if key not in seen:
            seen.add(key)
            out.append(it)
    if len(out) < len(items):
        print(f"  (removed {len(items) - len(out)} exact-duplicate record(s) from model response)")
    return out


def dedupe_by_keys(items: list, key_fields: list) -> list:
    """Narrower dedup for cases where two records can be 'the same row' despite
    differing on free-text fields the model can rephrase between repeats
    (device_group, offer_condition) — matching on those was letting real
    duplicates slip through. Used by the carrier-grid and device-offers
    extractors specifically."""
    seen, out = set(), []
    for it in items:
        sig = tuple(it.get(k) for k in key_fields)
        if sig not in seen:
            seen.add(sig)
            out.append(it)
    if len(out) < len(items):
        print(f"  (removed {len(items) - len(out)} exact-duplicate record(s) from model response)")
    return out


# ---- 2i. Logging -------------------------------------------------------------

def print_extraction_results(rows: list, review_rows: list, source_text: str, sample_ok: int = 3,
                              carrier_key: str = "carrier", item_key: str = "item_name",
                              amount_key: str = "offer_amount", tier_key: str = "plan_tier"):
    """Consistent logging: full detail (with inline source context) on
    everything flagged, a small sample of what looks fine, and a count
    summary. Takes review_rows so a row flagged for a reason OTHER than
    values_agree (e.g. new_carrier) still shows as flagged here — values_agree
    alone used to under-report this and made the OK/CHECK split misleading.

    carrier_key/item_key/amount_key let this work for tables whose raw row
    field names differ (e.g. retailer/device/discount_amount for retail
    readout, provider/metric_name/metric_value for business internet) —
    review_rows itself always uses the fixed carrier/item_name/offer_amount
    schema regardless of source, since that's what the review TABLE stores."""
    flagged_sigs = {(r.get("carrier"), r.get("item_name"), r.get("offer_amount")) for r in review_rows}

    def is_flagged(r):
        sig = (r.get(carrier_key), r.get(item_key), r.get(amount_key))
        return (not r.get("values_agree", True)) or (sig in flagged_sigs)

    ok_rows = [r for r in rows if not is_flagged(r)]
    check_rows = [r for r in rows if is_flagged(r)]
    print(f"  {len(rows)} record(s): {len(ok_rows)} OK, {len(check_rows)} flagged")

    if check_rows:
        normalized_source = re.sub(r"\s+", " ", source_text)
        print("  --- flagged, full detail ---")
        for r in check_rows:
            print(f"    [CHECK] {r.get(carrier_key)}: {r.get(item_key)} — {r.get(amount_key)!r}"
                  + (f"  (tier: {r[tier_key]})" if r.get(tier_key) else ""))
            num_match = re.search(r"[\d,]+", str(r.get(amount_key) or ""))
            idx = normalized_source.find(num_match.group()) if num_match else -1
            if idx == -1 and r.get(carrier_key):
                idx = normalized_source.find(str(r[carrier_key]))
            if idx == -1 and r.get(carrier_key):
                idx = normalized_source.find(str(r[carrier_key]).split()[0])  # e.g. "Cricket" vs "Cricket Wireless"
            if idx >= 0:
                print(f"       nearby source text: ...{normalized_source[max(0, idx - 40):idx + 150]}...")
            else:
                print("       (nothing matching found in source text at all)")

    if ok_rows:
        shown = ok_rows[:sample_ok]
        print(f"  --- sample of OK rows ({len(shown)} of {len(ok_rows)}) ---")
        for r in shown:
            print(f"    [OK] {r.get('carrier')}: {r.get('item_name')} — {r.get('offer_amount')}")


def print_generic_results(rows: list, review_rows: list, key_fields: list, sample: int = 5):
    """Lighter-weight logging for the five new (first-pass) page types below —
    their row shapes vary too much for one fixed field layout."""
    print(f"  {len(rows)} record(s), {len(review_rows)} flagged for review")
    for r in rows[:sample]:
        print("    " + " | ".join(f"{k}={r.get(k)!r}" for k in key_fields))
    if len(rows) > sample:
        print(f"    ... and {len(rows) - sample} more (full detail in the table, not the log)")


def print_flag_breakdown(review_rows: list):
    """Run-level summary of *why* things got flagged, visible without
    scrolling back through every page's detail."""
    if not review_rows:
        return
    counts = {}
    for r in review_rows:
        counts[r["flag_reason"]] = counts.get(r["flag_reason"], 0) + 1
    print("  Breakdown:", ", ".join(f"{reason}={n}" for reason, n in sorted(counts.items())))


# ============================================================================
# %% [Cell 8: Stage 1 bronze pipeline + upload widget]
# 3. STAGE 1 — BRONZE: PDF -> per-page text/image -> classified rows
# ============================================================================

# status: 'active' | 'superseded' | 'cancelled'
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{MANIFEST_TABLE}` (
        report_date DATE, run_id STRING, file_name STRING, file_hash STRING,
        status STRING,
        uploaded_at TIMESTAMP, uploaded_by STRING
    )
""").result()

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{BRONZE_TABLE}` (
        report_date DATE, run_id STRING, source_pdf STRING, page_number INT64,
        page_type STRING, text_extraction_raw STRING, llm_extraction_raw STRING,
        page_image_png BYTES, prompt_version STRING, extracted_at TIMESTAMP
    )
""").result()
bq.query(f"ALTER TABLE `{BRONZE_TABLE}` ADD COLUMN IF NOT EXISTS page_image_png BYTES").result()

# flag_reason: unclassified_page | new_page_type_auto_created | extraction_failed
#              | new_carrier | amount_mismatch
# status: pending | resolved
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{REVIEW_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, source_table STRING,
        flag_reason STRING,
        carrier STRING, item_name STRING, offer_amount STRING, detail STRING,
        status STRING,
        flagged_at TIMESTAMP
    )
""").result()

print("Manifest, bronze, and review tables ready.")


def run_extraction_pipeline(run_id: str, report_date, pdf_bytes: bytes, file_name: str):
    load_custom_patterns(force_refresh=True)  # pick up anything added since the last run
    print(f"[{run_id}] Opening {file_name} ...")
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    print(f"[{run_id}] {len(doc)} pages found.\n")

    rows = []
    total_pages = len(doc)
    for i, page in enumerate(doc):
        page_num = i + 1
        layout_text = reconstruct_layout_text(page)
        page_type, matched_pattern = classify_page(layout_text)

        if page_type == "unclassified" and page_num == 1:
            # Title-slide text is duplicated by a shadow-effect rendering (two
            # overlapping text boxes), which breaks literal-phrase matching
            # unpredictably. Position is the reliable signal, not the text.
            page_type, matched_pattern = "title_slide", "(fallback: first page of deck)"
        if page_type == "unclassified" and page_num == total_pages:
            # No text pattern is reliable here — full-bleed brand graphic, no
            # real title text. Position relative to deck length is safe to use.
            page_type, matched_pattern = "closing_slide", "(fallback: last page of deck)"

        registry_entry = PAGE_TYPE_REGISTRY.get(page_type, {"store_image": False, "stage2_strategy": "unhandled"})
        image_b64 = None
        stored_text = layout_text
        if registry_entry["store_image"]:
            image_b64 = base64.b64encode(render_page_image(page)).decode("ascii")
            # Column-aware reconstruction for anything that gets an image — this
            # is what actually feeds the Gemini prompt and the hallucination
            # guard downstream, so it should be the more reliable version.
            stored_text = reconstruct_columns(page)

        print(f"--- Page {page_num} ---")
        print(f"  matched pattern : {matched_pattern!r}")
        print(f"  page_type       : {page_type}  (strategy: {registry_entry['stage2_strategy']})")
        print(f"  image rendered  : {image_b64 is not None}")
        print(f"  text preview    : {stored_text[:120]!r}")
        print()

        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_pdf": file_name,
            "page_number": page_num, "page_type": page_type, "text_extraction_raw": stored_text,
            "llm_extraction_raw": None, "page_image_png": image_b64, "prompt_version": None,
            "extracted_at": datetime.utcnow().isoformat(),
        })

    unclassified = [r["page_number"] for r in rows if r["page_type"] == "unclassified"]
    if unclassified:
        print(f"[{run_id}] WARNING: {len(unclassified)} page(s) matched no known pattern: {unclassified}")
        print("Written to bronze as 'unclassified' — run auto_classify_unclassified_pages() in Stage 2, "
              "or inspect and add a pattern yourself via add_page_type_pattern().\n")

    try:
        write_rows(BRONZE_TABLE, rows)
    except Exception as e:
        print(f"[{run_id}] BigQuery write failed: {e}")
        return
    print(f"[{run_id}] Wrote {len(rows)} rows to bronze.")


# ---- Upload widget + explicit Upload button + dedup check + decision UI ----

upload_widget = widgets.FileUpload(accept=".pdf", multiple=False, description="Choose file")
upload_button = widgets.Button(description="Upload", button_style="primary", icon="upload")
output = widgets.Output()
display(widgets.HBox([upload_widget, upload_button]), output)

_last_pdf_bytes = None
_last_file_name = None


def proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode):
    with output:
        if mode == "cancel":
            print("Cancelled — nothing written.")
            return
        if mode == "replace":
            print(f"Deleting existing data for {report_date} ...")
            if not delete_existing_run(report_date):
                print("Aborting — old data wasn't fully cleared, so nothing new was written (avoids duplicates).")
                return
        write_manifest_row(report_date, run_id, file_name, hash_, status="active")
        run_extraction_pipeline(run_id, report_date, pdf_bytes, file_name)


def handle_upload(b):
    global _last_pdf_bytes, _last_file_name
    output.clear_output()
    if not upload_widget.value:
        with output:
            print("No file selected yet — choose a PDF first, then click Upload.")
        return

    uploaded = list(upload_widget.value.values())[0] if hasattr(upload_widget.value, "values") else upload_widget.value[0]
    file_name = uploaded["metadata"]["name"] if "metadata" in uploaded else uploaded.name
    pdf_bytes = uploaded["content"] if "content" in uploaded else uploaded.content
    _last_pdf_bytes, _last_file_name = pdf_bytes, file_name

    with output:
        report_date = parse_date_from_filename(file_name)
        if report_date is None:
            print(f"Couldn't parse a report date from filename '{file_name}' — check naming convention.")
            return

        match, found_date = verify_date_from_content(pdf_bytes, report_date)
        if match is False:
            print(f"Warning: filename implies {report_date}, but content suggests {found_date}. Proceeding with content date.")
            report_date = found_date

        hash_ = file_hash(pdf_bytes)
        existing = get_active_manifest_row(report_date)
        run_id = str(uuid.uuid4())

        if existing is None:
            print(f"No existing data for {report_date}. Proceeding.")
            proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode="new")
            return

        print(f"Data for {report_date} already exists (run {existing['run_id']}, file '{existing['file_name']}').")
        print("Choose how to proceed:")

        replace_btn = widgets.Button(description="Replace", button_style="danger")
        append_btn = widgets.Button(description="Append", button_style="warning")
        cancel_btn = widgets.Button(description="Cancel", button_style="")
        display(widgets.HBox([replace_btn, append_btn, cancel_btn]))

        replace_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "replace"))
        # NOTE on append: leaves two 'active' manifest rows for the same report_date
        # (nothing gets deleted). Decide before relying on this: should gold show
        # both runs, or should "append" actually behave like replace?
        append_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "append"))
        cancel_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "cancel"))


upload_button.on_click(handle_upload)


# ============================================================================
# %% [Cell 9: diagnostics]
# 4. DIAGNOSTICS
# ============================================================================

def inspect_page(page_number: int, method="columns", y_tolerance=3, pdf_bytes=None):
    """Full text reconstruction for one page, not just the 120-char bronze
    preview. method='columns' vs 'rows' — compare before deciding which a new
    page type actually needs."""
    pdf_bytes = pdf_bytes or _last_pdf_bytes
    if pdf_bytes is None:
        print("No PDF in memory yet — upload one first, or pass pdf_bytes explicitly.")
        return
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    page = doc[page_number - 1]
    text = reconstruct_columns(page, y_tolerance) if method == "columns" else reconstruct_layout_text(page, y_tolerance)
    print(text)


def dump_all_pages(pdf_bytes=None, y_tolerance=3):
    """Full-deck dump showing page_type + strategy + reconstructed text for
    every page, using the same registry that drives the real pipeline."""
    pdf_bytes = pdf_bytes or _last_pdf_bytes
    if pdf_bytes is None:
        print("No PDF in memory yet — upload one first.")
        return
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    total_pages = len(doc)

    for i, page in enumerate(doc):
        page_num = i + 1
        row_text = reconstruct_layout_text(page, y_tolerance)
        page_type, _ = classify_page(row_text)
        if page_type == "unclassified" and page_num == 1:
            page_type = "title_slide"
        if page_type == "unclassified" and page_num == total_pages:
            page_type = "closing_slide"

        registry_entry = PAGE_TYPE_REGISTRY.get(page_type, {"store_image": False, "stage2_strategy": "unhandled"})
        print(f"\n{'=' * 80}\nPAGE {page_num}  |  page_type: {page_type}  |  strategy: {registry_entry['stage2_strategy']}\n{'=' * 80}")

        if registry_entry["stage2_strategy"] == "skip":
            print("(non-content page, no extraction planned)")
            continue

        print(reconstruct_columns(page, y_tolerance) if registry_entry["store_image"] else row_text)


def show_bronze_state():
    print(f"[queried at {datetime.utcnow().isoformat()} UTC]\n")
    query = f"""
        SELECT run_id, report_date, page_type, COUNT(*) AS page_count
        FROM `{BRONZE_TABLE}`
        GROUP BY run_id, report_date, page_type
        ORDER BY report_date DESC, run_id, page_type
    """
    for row in bq.query(query).result():
        print(f"{row['report_date']}  |  run_id {row['run_id']}  |  {row['page_type']:35s}  |  {row['page_count']} page(s)")


def debug_amount_mismatches(run_id, page_number, table_id=None):
    """For any page with values_agree=False rows, show exactly what text those
    rows were checked against — direct inspection instead of guessing why a
    match failed. Defaults to the device offers table; pass table_id for
    another silver table."""
    table_id = table_id or SILVER_DEVICE_OFFERS_TABLE
    rows = list(bq.query(
        f"""SELECT carrier, item_name, offer_amount FROM `{table_id}`
            WHERE run_id = @run_id AND source_page = @page_number AND values_agree = FALSE""",
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
            bigquery.ScalarQueryParameter("page_number", "INT64", page_number),
        ]),
    ).result())
    bronze = list(bq.query(
        f"""SELECT text_extraction_raw FROM `{BRONZE_TABLE}`
            WHERE run_id = @run_id AND page_number = @page_number""",
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
            bigquery.ScalarQueryParameter("page_number", "INT64", page_number),
        ]),
    ).result())
    normalized = re.sub(r"\s+", " ", bronze[0]["text_extraction_raw"]) if bronze else ""

    for r in rows:
        amt = r["offer_amount"] or ""
        found = amt in normalized
        print(f"{r['carrier']:10s} {r['item_name']:20s} amount={amt!r:35s} literal_match={found}")
        if not found and r["carrier"]:
            idx = normalized.find(r["carrier"])
            print(f"   nearby context: ...{normalized[max(0, idx):idx + 250]}...")


# ============================================================================
# %% [Cell 10: ci_headlines + national_retail_readout]
# 5. STAGE 2 — TEXT-ONLY TIER
# ============================================================================

# ---- 5a. ci_headlines + national_retail_readout ----------------------------

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_CI_HEADLINES_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, carrier STRING,
        section STRING, headline_title STRING, headline_detail STRING,
        effective_date_range STRING, has_changes BOOL, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_RETAIL_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, week_ending STRING,
        retailer STRING, device STRING, discount_amount STRING, final_price STRING,
        carrier_service STRING, note STRING, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Silver ciHeadlines & nationalRetailReadout tables ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_CI_HEADLINES_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, carrier STRING,
        section STRING, headline_title STRING, headline_detail STRING,
        effective_date_range STRING, has_changes BOOL, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_RETAIL_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, week_ending STRING,
        retailer STRING, device STRING, discount_amount STRING, final_price STRING,
        carrier_service STRING, note STRING, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold ciHeadlines & nationalRetailReadout tables ready.")

CI_HEADLINES_SCHEMA = {
    "type": "object",
    "properties": {
        "headlines": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"}, "section": {"type": "string"},
                    "headline_title": {"type": "string"}, "headline_detail": {"type": "string"},
                    "effective_date_range": {"type": "string"}, "has_changes": {"type": "boolean"},
                },
                "required": ["carrier", "headline_title", "has_changes"],
            },
        }
    },
    "required": ["headlines"],
}

CI_HEADLINES_PROMPT = """You are extracting structured data from a competitive \
intelligence report page. The page contains a numbered list of carriers, each \
followed by either specific bulleted headline items describing changes, or a \
simple "No changes" note.

For EACH numbered carrier entry:
- If it says "No changes" (or equivalent), emit exactly one record with \
has_changes=false, headline_title="No changes", headline_detail="".
- Otherwise, emit one record per bullet point, with has_changes=true.

Extract "section" from the page's subheading (e.g. "Postpaid & Cable MVNO Key \
Highlights"). Extract any date or date range in parentheses into \
effective_date_range if present, otherwise leave it blank. Do not paraphrase or \
summarize headline_detail — keep it close to the source wording.

Known carriers — use these exact names when applicable: {known_carriers}

PAGE TEXT:
{page_text}"""

RETAIL_READOUT_SCHEMA = {
    "type": "object",
    "properties": {
        "week_ending": {"type": "string"},
        "retail_offers": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "retailer": {"type": "string"}, "device": {"type": "string"},
                    "discount_amount": {"type": "string"}, "final_price": {"type": "string"},
                    "carrier_service": {"type": "string"}, "note": {"type": "string"},
                },
                "required": ["retailer", "device"],
            },
        },
    },
    "required": ["retail_offers"],
}

RETAIL_READOUT_PROMPT = """You are extracting structured retail promotion data \
from a weekly competitive intelligence report page. The page is prose-style \
bullets describing which retailers are running which device discounts.

Extract "week_ending" from the page header. For each distinct retailer+device \
offer mentioned, extract one record with: retailer, device (model name only), \
discount_amount (as a dollar string, e.g. "$350"), final_price if mentioned, \
carrier_service if the offer is tied to a specific carrier/MVNO (otherwise \
"unlocked"), and a short note for any other relevant detail. If a bullet \
mentions no specific offer (e.g. "no new promotions"), skip it.

PAGE TEXT:
{page_text}"""


def extract_ci_headlines(page_number, page_text, report_date, run_id, known_carriers):
    result = call_gemini_json(
        CI_HEADLINES_PROMPT.format(page_text=page_text, known_carriers=", ".join(known_carriers)),
        CI_HEADLINES_SCHEMA)
    rows, review_rows = [], []
    for h in result.get("headlines", []):
        carrier = normalize_carrier(h.get("carrier"), known_carriers)
        # Only "carrier" is guarded, not headline_detail — narrative content
        # legitimately gets rephrased, forcing a literal match there would
        # just produce noise (same reasoning as postpaid_service_offers below).
        carrier_ok = values_present_in_text({"carrier": carrier}, ["carrier"], page_text)
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "carrier": carrier, "section": h.get("section"), "headline_title": h.get("headline_title"),
            "headline_detail": h.get("headline_detail"), "effective_date_range": h.get("effective_date_range"),
            "has_changes": h.get("has_changes"), "values_agree": carrier_ok,
            "llm_extraction_raw": json.dumps(result), "extracted_at": datetime.utcnow().isoformat(),
        })
        if carrier and carrier not in known_carriers:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_ciHeadlines", "flag_reason": "new_carrier", "carrier": carrier,
                "item_name": h.get("headline_title"), "offer_amount": None,
                "detail": f"'{carrier}' not in known carrier list.", "status": "pending",
                "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def extract_national_retail(page_number, page_text, report_date, run_id):
    result = call_gemini_json(RETAIL_READOUT_PROMPT.format(page_text=page_text), RETAIL_READOUT_SCHEMA)
    week_ending = result.get("week_ending")
    rows, review_rows = [], []
    for o in result.get("retail_offers", []):
        value_ok = values_present_in_text(o, ["retailer", "device", "discount_amount"], page_text)
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "week_ending": week_ending, "retailer": o.get("retailer"), "device": o.get("device"),
            "discount_amount": o.get("discount_amount"), "final_price": o.get("final_price"),
            "carrier_service": o.get("carrier_service"), "note": o.get("note"), "values_agree": value_ok,
            "llm_extraction_raw": json.dumps(result), "extracted_at": datetime.utcnow().isoformat(),
        })
        if not value_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_nationalRetailReadout", "flag_reason": "amount_mismatch",
                "carrier": o.get("carrier_service"), "item_name": o.get("device"), "offer_amount": o.get("discount_amount"),
                "detail": "retailer/device/discount_amount doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def run_text_only_extraction(run_id, report_date):
    print(f"[{run_id}] Text-only extraction stage starting...")
    known_carriers = get_known_carriers()
    ci_rows, ci_review, retail_rows, retail_review = [], [], [], []

    ci_pages = fetch_bronze_pages(run_id, ["ci_headlines"])
    retail_pages = fetch_bronze_pages(run_id, ["national_retail_readout"])
    if not ci_pages and not retail_pages:
        print(f"  No matching bronze pages found for run_id {run_id!r} — check show_bronze_state().")
        return

    for page in ci_pages:
        print(f"\n--- Page {page['page_number']} (ci_headlines) ---")
        try:
            rows, review = extract_ci_headlines(page["page_number"], page["text_extraction_raw"], report_date, run_id, known_carriers)
            print_generic_results(rows, review, ["carrier", "headline_title", "has_changes"])
            ci_rows.extend(rows); ci_review.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            ci_review.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_ciHeadlines", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None,
                "detail": traceback.format_exc()[-1500:], "status": "pending",
                "flagged_at": datetime.utcnow().isoformat(),
            })

    for page in retail_pages:
        print(f"\n--- Page {page['page_number']} (national_retail_readout) ---")
        try:
            rows, review = extract_national_retail(page["page_number"], page["text_extraction_raw"], report_date, run_id)
            print_extraction_results(rows, review, page["text_extraction_raw"],
                                      carrier_key="retailer", item_key="device", amount_key="discount_amount")
            retail_rows.extend(rows); retail_review.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            retail_review.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_nationalRetailReadout", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None,
                "detail": traceback.format_exc()[-1500:], "status": "pending",
                "flagged_at": datetime.utcnow().isoformat(),
            })

    if ci_rows:
        delete_silver_rows_for_date(SILVER_CI_HEADLINES_TABLE, report_date)
        write_rows(SILVER_CI_HEADLINES_TABLE, ci_rows)
        print(f"\n[{run_id}] Wrote {len(ci_rows)} rows to silver ciHeadlines (prior rows for this date cleared first).")
        gold_ci = promote_to_gold(GOLD_CI_HEADLINES_TABLE, ci_rows, report_date, known_carriers)
        print(f"[{run_id}] Promoted {len(gold_ci)} of {len(ci_rows)} rows to gold ciHeadlines.")
    if retail_rows:
        delete_silver_rows_for_date(SILVER_RETAIL_TABLE, report_date)
        write_rows(SILVER_RETAIL_TABLE, retail_rows)
        print(f"[{run_id}] Wrote {len(retail_rows)} rows to silver nationalRetailReadout (prior rows for this date cleared first).")
        gold_retail = promote_to_gold(GOLD_RETAIL_TABLE, retail_rows, report_date, known_carriers)
        print(f"[{run_id}] Promoted {len(gold_retail)} of {len(retail_rows)} rows to gold nationalRetailReadout.")
    all_review = ci_review + retail_review
    if all_review:
        write_review_rows(all_review)
        print(f"[{run_id}] Flagged {len(all_review)} row(s) for review.")
        print_flag_breakdown(all_review)
        warn_if_review_queue_too_large(all_review)


# %% [Cell 11: major_offer_changes]
# ---- 5b. major_offer_changes ------------------------------------------------

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_MAJOR_CHANGES_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, segment STRING,
        carrier STRING, change_type STRING, description STRING,
        effective_date_text STRING, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Silver majorOfferChanges table ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_MAJOR_CHANGES_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, segment STRING,
        carrier STRING, change_type STRING, description STRING,
        effective_date_text STRING, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold majorOfferChanges table ready.")

MAJOR_CHANGES_SCHEMA = {
    "type": "object",
    "properties": {
        "changes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "segment": {"type": "string"}, "carrier": {"type": "string"},
                    "change_type": {"type": "string"}, "description": {"type": "string"},
                    "effective_date_text": {"type": "string"},
                },
                "required": ["segment", "carrier", "change_type", "description"],
            },
        }
    },
    "required": ["changes"],
}

MAJOR_CHANGES_PROMPT = """This page lists what changed this week, organized as POSTPAID \
and PREPAID sections, each with carrier rows split into "OFFERS INTRODUCED" and "OFFERS \
REMOVED" columns.

Emit ONE RECORD PER BULLET POINT (or per carrier row if there's no bullet, e.g. a plain \
"No new offers" / "No changes" / "No expired offers" line — emit that too, it confirms \
this carrier was checked this week with nothing to report).

For each record: segment ("postpaid" or "prepaid"), carrier (use these exact names when \
applicable: {known_carriers}), change_type ("introduced" or "removed"), description (the \
bullet's own text), effective_date_text (whatever date/range annotation appears, e.g. \
"(6/18 - )" — leave blank if none, copy verbatim rather than reformatting).

TEXT:
{page_text}"""


def extract_major_changes(page_number, page_text, report_date, run_id, known_carriers):
    prompt = MAJOR_CHANGES_PROMPT.format(page_text=page_text, known_carriers=", ".join(known_carriers))
    result = call_gemini_json(prompt, MAJOR_CHANGES_SCHEMA)
    changes = dedupe_generic(result.get("changes", []))

    rows, review_rows = [], []
    for c in changes:
        carrier = normalize_carrier(c.get("carrier"), known_carriers)
        desc_ok = values_present_in_text(c, ["description"], page_text)
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "segment": c.get("segment"), "carrier": carrier, "change_type": c.get("change_type"),
            "description": c.get("description"), "effective_date_text": c.get("effective_date_text"),
            "values_agree": desc_ok, "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
        if not desc_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_majorOfferChanges", "flag_reason": "amount_mismatch",
                "carrier": carrier, "item_name": c.get("change_type"), "offer_amount": None,
                "detail": "description doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def run_major_changes_extraction(run_id, report_date):
    print(f"[{run_id}] Major offer changes extraction starting...")
    known_carriers = get_known_carriers()
    pages = fetch_bronze_pages(run_id, ["major_offer_changes"])
    if not pages:
        print("  No matching bronze pages found for this run_id.")
        return
    all_rows, all_review_rows = [], []
    for page in pages:
        print(f"\n--- Page {page['page_number']} ---")
        try:
            rows, review = extract_major_changes(page["page_number"], page["text_extraction_raw"], report_date, run_id, known_carriers)
            print_generic_results(rows, review, ["segment", "carrier", "change_type", "description"])
            all_rows.extend(rows); all_review_rows.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_majorOfferChanges", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": traceback.format_exc()[-1500:],  # full traceback, tail-truncated — not just the message
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    if all_rows:
        delete_silver_rows_for_date(SILVER_MAJOR_CHANGES_TABLE, report_date)
        write_rows(SILVER_MAJOR_CHANGES_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver majorOfferChanges (prior rows for this date cleared first).")
        gold_rows = promote_to_gold(GOLD_MAJOR_CHANGES_TABLE, all_rows, report_date, known_carriers)
        print(f"[{run_id}] Promoted {len(gold_rows)} of {len(all_rows)} rows to gold majorOfferChanges.")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)
        warn_if_review_queue_too_large(all_review_rows)


# %% [Cell 12: voice_plan_comparison]
# ---- 5c. voice_plan_comparison ----------------------------------------------

# audience: standard | switcher — derived from page number, not the model
# pricing_basis: the sub-table's own title, e.g. "Rack Rate", "Rack Rate less AutoPay discount and Promotions"
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_VOICE_PLAN_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64,
        audience STRING,
        pricing_basis STRING,
        lines INT64, tier STRING, carrier STRING, plan_name STRING, price FLOAT64,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Silver voicePlanComparison table ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_VOICE_PLAN_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64,
        audience STRING,
        pricing_basis STRING,
        lines INT64, tier STRING, carrier STRING, plan_name STRING, price FLOAT64,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold voicePlanComparison table ready.")

def detect_voice_plan_audience(page_text: str) -> str:
    """Derived from the page's own title text ('...for Switchers'), not raw
    page position. VOICE_PLAN_AUDIENCE_BY_PAGE (pages 23-28) was a real
    fragility: a slide added or removed anywhere earlier in a different
    week's deck shifts every page number after it, silently mislabeling
    audience without ever raising an error. Checked against the top of the
    page (where a title lives), not the whole page, to avoid a stray mention
    of the word elsewhere in a dense table."""
    return "switcher" if re.search(r"\bswitcher", page_text[:400], re.IGNORECASE) else "standard"


VOICE_PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "rows": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "pricing_basis": {"type": "string"}, "lines": {"type": "integer"},
                    "tier": {"type": "string"}, "carrier": {"type": "string"},
                    "plan_name": {"type": "string"}, "price": {"type": "number"},
                },
                "required": ["pricing_basis", "lines", "tier", "carrier", "plan_name", "price"],
            },
        }
    },
    "required": ["rows"],
}

VOICE_PLAN_PROMPT = """This page shows one or two pricing tables comparing T-Mobile, \
AT&T, and Verizon business voice plans by line count (1-12 or 6-12), each split into \
Good/Better/Best tiers with a specific named plan per carrier per tier (e.g. AT&T \
"Standard 3.0", Verizon "My Biz", T-Mobile "CoreMobile" for Good; different plan names \
for Better and Best).

The IMAGE shows the actual table layout — use it to correctly read each cell, since \
this is a dense numeric grid where text-only extraction has been unreliable. The text \
below is a secondary reference for confirming exact figures.

If the page has two tables (a "Rack Rate" full-price one and a separate "Rack Rate less \
AutoPay discount and Promotions" one), extract BOTH, and set pricing_basis to each \
table's own title exactly as printed. If there's only one table, use its title for every row.

For every (lines, tier, carrier) cell with a price, emit one record: pricing_basis, \
lines, tier (Good/Better/Best), carrier, plan_name (that carrier's specific plan name \
for this tier), price (the number only, no $ sign).

REFERENCE TEXT:
{page_text}"""


def extract_voice_plan_comparison(page_number, page_text, image_bytes, report_date, run_id):
    result = call_gemini_json_with_image(VOICE_PLAN_PROMPT.format(page_text=page_text), VOICE_PLAN_SCHEMA, image_bytes)
    rows_raw = dedupe_generic(result.get("rows", []))
    audience = detect_voice_plan_audience(page_text)

    rows, review_rows = [], []
    for r in rows_raw:
        price_ok = numeric_present_in_text(r.get("price"), page_text)
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "audience": audience, "pricing_basis": r.get("pricing_basis"), "lines": r.get("lines"),
            "tier": r.get("tier"), "carrier": r.get("carrier"), "plan_name": r.get("plan_name"),
            "price": r.get("price"), "values_agree": price_ok, "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
        if not price_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_voicePlanComparison", "flag_reason": "amount_mismatch",
                "carrier": r.get("carrier"), "item_name": r.get("plan_name"), "offer_amount": str(r.get("price")),
                "detail": f"Price {r.get('price')} not found among dollar figures on this page.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def run_voice_plan_extraction(run_id, report_date):
    print(f"[{run_id}] Voice plan comparison extraction starting...")
    pages = fetch_bronze_pages_with_image(run_id, ["voice_plan_comparison"])
    if not pages:
        print("  No matching bronze pages found for this run_id — fine if this week's deck "
              "doesn't include this page type, see run_extraction_pipeline's classification log.")
        return
    all_rows, all_review_rows = [], []
    for page in pages:
        print(f"\n--- Page {page['page_number']} ---")
        if not page["page_image_png"]:
            print("  SKIPPED: no image stored for this page (check PAGE_TYPE_REGISTRY store_image for this type).")
            continue
        try:
            rows, review = extract_voice_plan_comparison(page["page_number"], page["text_extraction_raw"], page["page_image_png"], report_date, run_id)
            print_generic_results(rows, review, ["pricing_basis", "lines", "tier", "carrier", "price"])
            all_rows.extend(rows); all_review_rows.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_voicePlanComparison", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": traceback.format_exc()[-1500:],
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    if all_rows:
        delete_silver_rows_for_date(SILVER_VOICE_PLAN_TABLE, report_date)
        write_rows(SILVER_VOICE_PLAN_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver voicePlanComparison (prior rows for this date cleared first).")
        gold_rows = promote_to_gold(GOLD_VOICE_PLAN_TABLE, all_rows, report_date)
        print(f"[{run_id}] Promoted {len(gold_rows)} of {len(all_rows)} rows to gold voicePlanComparison.")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)
        warn_if_review_queue_too_large(all_review_rows)


# %% [Cell 13: business_internet_comparison]
# ---- 5d. business_internet_comparison ---------------------------------------

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_BIZ_INTERNET_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, provider STRING,
        plan_tier STRING, metric_name STRING, metric_value STRING,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Silver businessInternetComparison table ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_BIZ_INTERNET_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, provider STRING,
        plan_tier STRING, metric_name STRING, metric_value STRING,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold businessInternetComparison table ready.")

BIZ_INTERNET_SCHEMA = {
    "type": "object",
    "properties": {
        "rows": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "provider": {"type": "string"}, "plan_tier": {"type": "string"},
                    "metric_name": {"type": "string"}, "metric_value": {"type": "string"},
                },
                "required": ["provider", "plan_tier", "metric_name", "metric_value"],
            },
        }
    },
    "required": ["rows"],
}

BIZ_INTERNET_PROMPT = """This page compares business internet plans across T-Mobile, \
AT&T, and Verizon, where each provider has multiple plan tiers as separate columns \
(e.g. T-Mobile: Small Business, Grow Business; AT&T: Advanced, Standard/Premium; \
Verizon: 100 Mbps, 200 Mbps), and each row is a spec/metric (Bandwidth, Data, Rack \
Rate, AutoPay Discount, Bundle Credit, Promotional Rate, ETF/Switch Credit, Price \
Guarantee, Free Trial Period, Device Included, Install Type, Max Credit per Line, etc).

The IMAGE shows the actual table layout — use it to correctly associate each metric \
row with the right provider/tier column. The text below is a secondary reference for \
confirming exact wording.

For every provider + plan_tier + metric cell, emit one record: provider, plan_tier \
(that column's specific tier name), metric_name (the row label), metric_value (exactly \
as printed, including "Not Stated" where that's what's shown).

REFERENCE TEXT:
{page_text}"""


def extract_biz_internet_comparison(page_number, page_text, image_bytes, report_date, run_id):
    result = call_gemini_json_with_image(BIZ_INTERNET_PROMPT.format(page_text=page_text), BIZ_INTERNET_SCHEMA, image_bytes)
    rows_raw = dedupe_generic(result.get("rows", []))
    rows, review_rows = [], []
    for r in rows_raw:
        value_ok = values_present_in_text(r, ["metric_value"], page_text)
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "provider": r.get("provider"), "plan_tier": r.get("plan_tier"), "metric_name": r.get("metric_name"),
            "metric_value": r.get("metric_value"), "values_agree": value_ok, "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
        if not value_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_businessInternetComparison", "flag_reason": "amount_mismatch",
                "carrier": r.get("provider"), "item_name": r.get("metric_name"), "offer_amount": r.get("metric_value"),
                "detail": "metric_value doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def run_biz_internet_extraction(run_id, report_date):
    print(f"[{run_id}] Business internet comparison extraction starting...")
    pages = fetch_bronze_pages_with_image(run_id, ["business_internet_comparison"])
    if not pages:
        print("  No matching bronze pages found for this run_id — fine if this week's deck "
              "doesn't include this page type, see run_extraction_pipeline's classification log.")
        return
    all_rows, all_review_rows = [], []
    for page in pages:
        print(f"\n--- Page {page['page_number']} ---")
        if not page["page_image_png"]:
            print("  SKIPPED: no image stored for this page (check PAGE_TYPE_REGISTRY store_image for this type).")
            continue
        try:
            rows, review = extract_biz_internet_comparison(page["page_number"], page["text_extraction_raw"], page["page_image_png"], report_date, run_id)
            print_extraction_results(rows, review, page["text_extraction_raw"],
                                      carrier_key="provider", item_key="metric_name", amount_key="metric_value")
            all_rows.extend(rows); all_review_rows.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_businessInternetComparison", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": traceback.format_exc()[-1500:],
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    if all_rows:
        delete_silver_rows_for_date(SILVER_BIZ_INTERNET_TABLE, report_date)
        write_rows(SILVER_BIZ_INTERNET_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver businessInternetComparison (prior rows for this date cleared first).")
        gold_rows = promote_to_gold(GOLD_BIZ_INTERNET_TABLE, all_rows, report_date)
        print(f"[{run_id}] Promoted {len(gold_rows)} of {len(all_rows)} rows to gold businessInternetComparison.")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)
        warn_if_review_queue_too_large(all_review_rows)


# %% [Cell 14: prepaid_price_grid_detail]
# ---- 5e. prepaid_price_grid_detail -------------------------------------------
# Highest-risk of the text-only group: a single carrier (no carrier dimension
# needed) but a genuinely nested column header (connection type x registration
# type x ID requirement x plan price tier). Scoped to the MAIN phone grid only
# — the BTS/watch/tablet sub-table lower on the same page has a structurally
# different column set and is deliberately out of scope here.

# connection_type: e.g. Always Connected Upgrade, Level 1/2/3 Metro Annual Upgrade, "4 for $100" Non-Port
# registration_type: New Number | Port
# id_requirement: No ID | w/ ID
# plan_price_tier: $40 | $50 | $60 | Period
# device_price: FREE | dollar figure | N/A — kept as string, not force-numeric
# device_price_numeric: parsed for convenience — FREE->0.0, dollar figure->that number, else NULL
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_PREPAID_GRID_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, device_name STRING,
        retail_price STRING,
        connection_type STRING,
        registration_type STRING,
        id_requirement STRING,
        plan_price_tier STRING,
        device_price STRING,
        device_price_numeric FLOAT64,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Silver prepaidPriceGridDetail table ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_PREPAID_GRID_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, device_name STRING,
        retail_price STRING,
        connection_type STRING,
        registration_type STRING,
        id_requirement STRING,
        plan_price_tier STRING,
        device_price STRING,
        device_price_numeric FLOAT64,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold prepaidPriceGridDetail table ready.")

PREPAID_GRID_SCHEMA = {
    "type": "object",
    "properties": {
        "rows": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "device_name": {"type": "string"}, "retail_price": {"type": "string"},
                    "connection_type": {"type": "string"}, "registration_type": {"type": "string"},
                    "id_requirement": {"type": "string"}, "plan_price_tier": {"type": "string"},
                    "device_price": {"type": "string"},
                },
                "required": ["device_name", "connection_type", "registration_type", "plan_price_tier", "device_price"],
            },
        }
    },
    "required": ["rows"],
}

PREPAID_GRID_PROMPT = """This is the Metro by T-Mobile phone price grid ONLY — ignore any \
separate BTS/watch/tablet/hotspot table lower on the page, that is out of scope here.

The grid has one row per device (with its retail price), and many price columns grouped \
under: connection_type (Always Connected Upgrade, Level 1/2/3 Metro Annual Upgrade, or \
"4 for $100" Non-Port), then registration_type (New Number or Port), then id_requirement \
(No ID or w/ ID), then plan_price_tier ($40, $50, $60, or a flat "Period" rate).

For EVERY device x column-combination cell that has a price (including "FREE"), emit one \
record: device_name, retail_price (the device's base retail price), connection_type, \
registration_type, id_requirement, plan_price_tier, device_price (exactly as shown: \
"FREE", a dollar figure, or "N/A"). Skip cells that are genuinely blank.

TEXT:
{page_text}"""


def parse_device_price_numeric(text):
    if not text:
        return None
    if str(text).strip().upper() == "FREE":
        return 0.0
    return extract_numeric_value(text)


def extract_prepaid_grid(page_number, page_text, report_date, run_id):
    result = call_gemini_json(PREPAID_GRID_PROMPT.format(page_text=page_text), PREPAID_GRID_SCHEMA)
    rows_raw = dedupe_generic(result.get("rows", []))
    rows, review_rows = [], []
    for r in rows_raw:
        value_ok = values_present_in_text(r, ["device_price"], page_text)
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "device_name": r.get("device_name"), "retail_price": r.get("retail_price"),
            "connection_type": r.get("connection_type"), "registration_type": r.get("registration_type"),
            "id_requirement": r.get("id_requirement"), "plan_price_tier": r.get("plan_price_tier"),
            "device_price": r.get("device_price"),
            "device_price_numeric": parse_device_price_numeric(r.get("device_price")),
            "values_agree": value_ok, "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
        if not value_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_prepaidPriceGridDetail", "flag_reason": "amount_mismatch",
                "carrier": "Metro by T-Mobile", "item_name": r.get("device_name"), "offer_amount": r.get("device_price"),
                "detail": "device_price doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def run_prepaid_grid_extraction(run_id, report_date):
    print(f"[{run_id}] Prepaid price grid extraction starting...")
    pages = fetch_bronze_pages(run_id, ["prepaid_price_grid_detail"])
    if not pages:
        print("  No matching bronze pages found for this run_id.")
        return
    all_rows, all_review_rows = [], []
    for page in pages:
        print(f"\n--- Page {page['page_number']} ---")
        try:
            rows, review = extract_prepaid_grid(page["page_number"], page["text_extraction_raw"], report_date, run_id)
            print_generic_results(rows, review, ["device_name", "connection_type", "registration_type", "plan_price_tier", "device_price"])
            all_rows.extend(rows); all_review_rows.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_prepaidPriceGridDetail", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": traceback.format_exc()[-1500:],  # full traceback, tail-truncated — not just the message
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    if all_rows:
        delete_silver_rows_for_date(SILVER_PREPAID_GRID_TABLE, report_date)
        write_rows(SILVER_PREPAID_GRID_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver prepaidPriceGridDetail (prior rows for this date cleared first).")
        gold_rows = promote_to_gold(GOLD_PREPAID_GRID_TABLE, all_rows, report_date)
        print(f"[{run_id}] Promoted {len(gold_rows)} of {len(all_rows)} rows to gold prepaidPriceGridDetail.")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)
        warn_if_review_queue_too_large(all_review_rows)


# %% [Cell 15: postpaid_service_offers]
# ---- 5f. postpaid_service_offers ---------------------------------------------
# Narrative/feature content, not discrete numeric offers. No strict guard on
# "detail" — same reasoning as ci_headlines: narrative content legitimately
# gets paraphrased, so forcing a literal match here would just produce noise.

# category: e.g. BYOD Wireless Service, Target Segment Offers, Value-Added Services, 5G News, Home Internet
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_SERVICE_OFFERS_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, carrier STRING,
        category STRING,
        feature_name STRING, detail STRING, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Silver postpaidServiceOffers table ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_SERVICE_OFFERS_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64, carrier STRING,
        category STRING,
        feature_name STRING, detail STRING, values_agree BOOL,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold postpaidServiceOffers table ready.")

SERVICE_OFFERS_SCHEMA = {
    "type": "object",
    "properties": {
        "rows": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"}, "category": {"type": "string"},
                    "feature_name": {"type": "string"}, "detail": {"type": "string"},
                },
                "required": ["carrier", "category", "feature_name", "detail"],
            },
        }
    },
    "required": ["rows"],
}

SERVICE_OFFERS_PROMPT = """This page describes service-level features (not device offers) \
by carrier, organized under column headers like Brand, BYOD, Wireless Service, Target \
Segment Offers, Value-Added Services, 5G News, and (for Verizon) Home Internet.

For each distinct feature/bullet under a carrier's column, emit one record: carrier, \
category (the column header it's under, exactly as printed), feature_name (a short label \
for the specific feature), detail (the fuller description in your own words, condensed — \
paraphrasing is fine and expected here, unlike dollar-figure fields elsewhere).

Known carriers — use these exact names when applicable: {known_carriers}

TEXT:
{page_text}"""


def extract_service_offers(page_number, page_text, report_date, run_id, known_carriers):
    prompt = SERVICE_OFFERS_PROMPT.format(page_text=page_text, known_carriers=", ".join(known_carriers))
    result = call_gemini_json(prompt, SERVICE_OFFERS_SCHEMA)
    rows_raw = dedupe_generic(result.get("rows", []))
    rows = []
    for r in rows_raw:
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "carrier": normalize_carrier(r.get("carrier"), known_carriers), "category": r.get("category"),
            "feature_name": r.get("feature_name"), "detail": r.get("detail"),
            "values_agree": True,  # no strict guard by design — see module docstring above
            "llm_extraction_raw": json.dumps(result), "extracted_at": datetime.utcnow().isoformat(),
        })
    return rows, []  # nothing here is a dollar-figure claim to verify


def run_service_offers_extraction(run_id, report_date):
    print(f"[{run_id}] Postpaid service offers extraction starting...")
    known_carriers = get_known_carriers()
    pages = fetch_bronze_pages(run_id, ["postpaid_service_offers"])
    if not pages:
        print("  No matching bronze pages found for this run_id.")
        return
    all_rows, all_review_rows = [], []
    for page in pages:
        print(f"\n--- Page {page['page_number']} ---")
        try:
            rows, _ = extract_service_offers(page["page_number"], page["text_extraction_raw"], report_date, run_id, known_carriers)
            print_generic_results(rows, [], ["carrier", "category", "feature_name"])
            all_rows.extend(rows)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_postpaidServiceOffers", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None,
                "detail": traceback.format_exc()[-1500:], "status": "pending",
                "flagged_at": datetime.utcnow().isoformat(),
            })
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        warn_if_review_queue_too_large(all_review_rows)
    if all_rows:
        delete_silver_rows_for_date(SILVER_SERVICE_OFFERS_TABLE, report_date)
        write_rows(SILVER_SERVICE_OFFERS_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver postpaidServiceOffers (prior rows for this date cleared first).")
        gold_rows = promote_to_gold(GOLD_SERVICE_OFFERS_TABLE, all_rows, report_date, known_carriers)
        print(f"[{run_id}] Promoted {len(gold_rows)} of {len(all_rows)} rows to gold postpaidServiceOffers.")


# ============================================================================
# %% [Cell 16: text-first grid tier (bts/cable_mvno/prepaid_brand)]
# 6. STAGE 2 — TEXT-FIRST GRID TIER
#    postpaid_bts_offers, cable_mvno_offers, prepaid_brand_offers: same carrier
#    x item x offer shape as device offers, but text alone proved adequate
#    (unlike business_device_offers — see Section 7 for why that one escalated).
# ============================================================================

# page_type: which section this came from
# item_group: other items that shared this exact offer (like device_group)
# extraction_method: 'text' or 'text+image' — tracks which pages needed escalation
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_GRID_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64,
        page_type STRING,
        carrier STRING, item_name STRING,
        item_group STRING,
        plan_tier STRING, offer_type STRING, offer_type_raw STRING, offer_amount STRING,
        offer_amount_numeric FLOAT64, requires_trade_in BOOL, requires_port BOOL,
        is_recurring_credit BOOL, credit_monthly_amount FLOAT64, credit_duration_months INT64,
        offer_condition STRING,
        values_agree BOOL,
        extraction_method STRING,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
for col, typ in [("offer_amount_numeric", "FLOAT64"), ("requires_trade_in", "BOOL"), ("requires_port", "BOOL"),
                  ("is_recurring_credit", "BOOL"), ("credit_monthly_amount", "FLOAT64"), ("credit_duration_months", "INT64"),
                  ("offer_type_raw", "STRING")]:
    bq.query(f"ALTER TABLE `{SILVER_GRID_TABLE}` ADD COLUMN IF NOT EXISTS {col} {typ}").result()
print("Silver carrierOfferGrid table ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_GRID_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64,
        page_type STRING,
        carrier STRING, item_name STRING,
        item_group STRING,
        plan_tier STRING, offer_type STRING, offer_type_raw STRING, offer_amount STRING,
        offer_amount_numeric FLOAT64, requires_trade_in BOOL, requires_port BOOL,
        is_recurring_credit BOOL, credit_monthly_amount FLOAT64, credit_duration_months INT64,
        offer_condition STRING,
        values_agree BOOL,
        extraction_method STRING,
        llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold carrierOfferGrid table ready.")

GRID_TEXT_FIRST_PAGE_TYPES = [
    pt for pt, cfg in PAGE_TYPE_REGISTRY.items() if cfg["stage2_strategy"] == "text_first_grid"
]

# Canonical offer_type categories — built from an actual distinct-value pull
# against real silver data (SELECT offer_type, COUNT(*) ... GROUP BY offer_type),
# not a guess. This list is enforced via a JSON schema enum below, not a
# post-hoc lookup table: the model literally cannot emit anything outside it,
# so case/wording drift (the 'Discount'/'discount'/'Trade-in credit' mess a
# free-text field produced) becomes structurally impossible going forward,
# not just cleaned up after the fact. "other" is the deliberate escape hatch —
# offer_type_raw always carries what the model actually saw, so a genuinely
# new offer type is visible (flagged to review) rather than either forced
# into a bad-fit bucket or silently lost.
OFFER_TYPE_CATEGORIES = [
    "discount", "trade-in credit", "bill credit", "eip buyout", "bogo",
    "free device", "gift card", "bundle discount", "multi-line discount",
    "port-in credit", "plan pricing", "other",
]

# Shared across both "offer" extractors (this section and Section 7's image
# tier) — extra structured fields beyond the raw offer_amount string, so
# querying doesn't require re-parsing free text like "$400 Credit ($10/mo. x
# 36 mos)" downstream. None of these are in any schema's "required" list —
# the model omits whichever don't apply, and BigQuery stores that as NULL,
# rather than forcing a guessed value.
OFFER_ENRICHMENT_SCHEMA_FIELDS = {
    "offer_type": {"type": "string", "enum": OFFER_TYPE_CATEGORIES},
    "offer_type_raw": {"type": "string"},
    "offer_amount_numeric": {"type": "number"},
    "requires_trade_in": {"type": "boolean"},
    "requires_port": {"type": "boolean"},
    "is_recurring_credit": {"type": "boolean"},
    "credit_monthly_amount": {"type": "number"},
    "credit_duration_months": {"type": "integer"},
}

OFFER_ENRICHMENT_PROMPT_FRAGMENT = """
Additionally, for each record also extract these (omit any that don't apply — \
don't guess, leave them out rather than force a value):
- offer_type: choose the CLOSEST match from this fixed list, based on what the \
offer actually is, not its exact wording on the page: {offer_type_categories}. \
If genuinely none of these fit, use "other" — don't force a bad match.
- offer_type_raw: a short label in your own words for what kind of offer this \
is, based on how the source describes it. Always populate this, even when \
offer_type above is a clean, confident match — this is what lets a new offer \
type get noticed later even when "other" wasn't the right call for it.
- offer_amount_numeric: the dollar amount as a plain number, no "$" and no "off"/ \
"credit" wording (e.g. "$1,100 off" -> 1100, "$0.99/mo." -> 0.99). Omit if there's \
no single clear dollar figure (e.g. "On Us", "FREE" -> omit, unless the source \
explicitly states "$0").
- requires_trade_in: true if the offer's condition requires a device trade-in.
- requires_port: true if the offer's condition requires porting in from another carrier.
- is_recurring_credit: true if the offer is a recurring monthly bill credit spread \
over time (e.g. "$10/mo. x 36 mos"), as opposed to a one-time discount, EIP payoff, \
or upfront price.
- credit_monthly_amount: if is_recurring_credit, the per-month amount as a plain number.
- credit_duration_months: if is_recurring_credit, the number of months the credit lasts.
"""
OFFER_ENRICHMENT_PROMPT_FRAGMENT = OFFER_ENRICHMENT_PROMPT_FRAGMENT.format(
    offer_type_categories=", ".join(OFFER_TYPE_CATEGORIES)
)

CARRIER_GRID_SCHEMA = {
    "type": "object",
    "properties": {
        "offers": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"}, "item_name": {"type": "string"},
                    "item_group": {"type": "string"}, "plan_tier": {"type": "string"},
                    "offer_amount": {"type": "string"},
                    "offer_condition": {"type": "string"},
                    **OFFER_ENRICHMENT_SCHEMA_FIELDS,
                },
                "required": ["carrier", "item_name", "item_group"],
            },
        }
    },
    "required": ["offers"],
}

CARRIER_GRID_TEXT_PROMPT = """You are extracting structured offer data from a carrier \
comparison table. There is no image for this page — work from the text below, which \
is a column-aware reconstruction (each carrier's content is grouped together, marked \
by [COLUMN N] where applicable).

IMPORTANT — two patterns seen across this report apply here too:
1. One offer can apply to several items at once (e.g. one bullet covering several \
watch models or device tiers). Emit ONE RECORD PER ITEM, repeating the shared values, \
and set item_group to the full comma-separated list of items that share it (including \
the item itself). For a standalone offer, item_group should just equal item_name.
2. One item can have different prices per rate plan tier (e.g. Good/Better/Best). Emit \
ONE RECORD PER TIER, with plan_tier set to that tier's name and offer_amount set to \
only that tier's price. Do not combine multiple tier prices into one offer_amount string.

Known carriers — use these exact names when the offer is from one of them, for naming \
consistency across weeks: {known_carriers}

For each carrier + item (+ tier, if applicable) combination with a stated offer, \
extract one record: carrier, item_name (exactly one item), item_group, plan_tier (if \
applicable, else blank), offer_type (your best label), offer_amount (for exactly this \
one record), offer_condition (the requirement text). Skip entries with no stated offer \
("No offer", "--", "N/A").
{enrichment_fragment}
TEXT:
{page_text}"""

CARRIER_GRID_DEDUPE_KEYS = ["carrier", "item_name", "plan_tier", "offer_amount"]


def extract_carrier_grid_text(page_number, page_type, page_text, report_date, run_id, known_carriers):
    prompt = CARRIER_GRID_TEXT_PROMPT.format(page_text=page_text, known_carriers=", ".join(known_carriers),
                                              enrichment_fragment=OFFER_ENRICHMENT_PROMPT_FRAGMENT)
    result = call_gemini_json(prompt, CARRIER_GRID_SCHEMA)
    deduped_offers = dedupe_by_keys(result.get("offers", []), CARRIER_GRID_DEDUPE_KEYS)

    rows, review_rows = [], []
    for o in deduped_offers:
        carrier = normalize_carrier(o.get("carrier"), known_carriers)
        amount_ok = values_present_in_text(o, ["offer_amount"], page_text)
        offer_type = o.get("offer_type")  # schema-enum-constrained now — no post-hoc
                                            # normalization needed, the model can't drift
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "page_type": page_type, "carrier": carrier, "item_name": o.get("item_name"),
            "item_group": o.get("item_group"), "plan_tier": o.get("plan_tier"),
            "offer_type": offer_type, "offer_type_raw": o.get("offer_type_raw"),
            "offer_amount": o.get("offer_amount"),
            "offer_amount_numeric": o.get("offer_amount_numeric"),
            "requires_trade_in": o.get("requires_trade_in"), "requires_port": o.get("requires_port"),
            "is_recurring_credit": o.get("is_recurring_credit"),
            "credit_monthly_amount": o.get("credit_monthly_amount"),
            "credit_duration_months": o.get("credit_duration_months"),
            "offer_condition": o.get("offer_condition"),
            "values_agree": amount_ok, "extraction_method": "text", "llm_extraction_raw": json.dumps(result),
            "extracted_at": datetime.utcnow().isoformat(),
        })
        if offer_type == "other":
            # Independent of the other checks below — a new offer type is worth
            # surfacing even on a row that's otherwise perfectly clean, and
            # shouldn't get hidden behind whichever flag reason happens to fire first.
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_carrierOfferGrid", "flag_reason": "new_offer_type",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": f"offer_type_raw: {o.get('offer_type_raw')!r} — didn't fit any known category.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
        if carrier and carrier not in known_carriers:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_carrierOfferGrid", "flag_reason": "new_carrier",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": f"'{carrier}' not in known carrier list.", "status": "pending",
                "flagged_at": datetime.utcnow().isoformat(),
            })
        elif not amount_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_carrierOfferGrid", "flag_reason": "amount_mismatch",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": "offer_amount doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def run_carrier_grid_extraction(run_id, report_date):
    print(f"[{run_id}] Carrier grid (text-first) extraction starting...")
    known_carriers = get_known_carriers()
    pages = fetch_bronze_pages(run_id, GRID_TEXT_FIRST_PAGE_TYPES)
    if not pages:
        print("  No matching bronze pages found for this run_id — check show_bronze_state().")
        return
    all_rows, all_review_rows = [], []
    for page in pages:
        print(f"\n--- Page {page['page_number']} ({page['page_type']}) ---")
        try:
            rows, review = extract_carrier_grid_text(page["page_number"], page["page_type"], page["text_extraction_raw"], report_date, run_id, known_carriers)
            print(f"  extracted {len(rows)} records ({len(review)} flagged for review)")
            print_extraction_results(rows, review, page["text_extraction_raw"])
            all_rows.extend(rows); all_review_rows.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_carrierOfferGrid", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": traceback.format_exc()[-1500:],  # full traceback, tail-truncated — not just the message
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    if all_rows:
        delete_silver_rows_for_date(SILVER_GRID_TABLE, report_date)
        write_rows(SILVER_GRID_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver carrierOfferGrid (prior rows for this date cleared first).")
        gold_rows = promote_to_gold(GOLD_GRID_TABLE, all_rows, report_date, known_carriers)
        print(f"[{run_id}] Promoted {len(gold_rows)} of {len(all_rows)} rows to gold carrierOfferGrid.")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)
        warn_if_review_queue_too_large(all_review_rows)


# ============================================================================
# %% [Cell 17: image tier (device offers incl. business)]
# 7. STAGE 2 — IMAGE TIER
#    postpaid_device_offers_apple/samsung_google/motorola_affordable, and
#    business_device_offers (moved here after a confirmed cross-column
#    attribution error: that page reuses tier labels — SuperMobile/ProMobile/
#    CoreMobile, "Adv Unl or higher" — across multiple different figures
#    within one carrier's column, which text-only got wrong for Verizon).
# ============================================================================

# device_category: apple | samsung_google | motorola_affordable | business
# device_group: comma-separated devices that shared this offer, incl. item_name
# plan_tier: e.g. Good/Better/Best when pricing varies by rate plan tier
bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{SILVER_DEVICE_OFFERS_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64,
        device_category STRING,
        carrier STRING, item_name STRING,
        device_group STRING,
        plan_tier STRING,
        offer_type STRING, offer_type_raw STRING, offer_amount STRING,
        offer_amount_numeric FLOAT64, requires_trade_in BOOL, requires_port BOOL,
        is_recurring_credit BOOL, credit_monthly_amount FLOAT64, credit_duration_months INT64,
        offer_condition STRING,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
for col, typ in [("offer_amount_numeric", "FLOAT64"), ("requires_trade_in", "BOOL"), ("requires_port", "BOOL"),
                  ("is_recurring_credit", "BOOL"), ("credit_monthly_amount", "FLOAT64"), ("credit_duration_months", "INT64"),
                  ("offer_type_raw", "STRING")]:
    bq.query(f"ALTER TABLE `{SILVER_DEVICE_OFFERS_TABLE}` ADD COLUMN IF NOT EXISTS {col} {typ}").result()
print("Silver postpaidDeviceOffers table ready.")

bq.query(f"""
    CREATE TABLE IF NOT EXISTS `{GOLD_DEVICE_OFFERS_TABLE}` (
        report_date DATE, run_id STRING, source_page INT64,
        device_category STRING,
        carrier STRING, item_name STRING,
        device_group STRING,
        plan_tier STRING,
        offer_type STRING, offer_type_raw STRING, offer_amount STRING,
        offer_amount_numeric FLOAT64, requires_trade_in BOOL, requires_port BOOL,
        is_recurring_credit BOOL, credit_monthly_amount FLOAT64, credit_duration_months INT64,
        offer_condition STRING,
        values_agree BOOL, llm_extraction_raw STRING, extracted_at TIMESTAMP
    )
""").result()
print("Gold postpaidDeviceOffers table ready.")

DEVICE_CATEGORY_MAP = {
    pt: pt.replace("postpaid_device_offers_", "").replace("business_device_offers", "business")
    for pt, cfg in PAGE_TYPE_REGISTRY.items() if cfg["stage2_strategy"] == "image"
}

GRID_OFFERS_SCHEMA = {
    "type": "object",
    "properties": {
        "offers": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "carrier": {"type": "string"}, "item_name": {"type": "string"},
                    "device_group": {"type": "string"}, "plan_tier": {"type": "string"},
                    "offer_amount": {"type": "string"},
                    "offer_condition": {"type": "string"},
                    **OFFER_ENRICHMENT_SCHEMA_FIELDS,
                },
                "required": ["carrier", "item_name", "device_group"],
            },
        }
    },
    "required": ["offers"],
}

GRID_OFFERS_PROMPT = """You are extracting structured offer data from a table on \
this page. The IMAGE shows the actual visual table layout — use it to correctly \
associate each carrier's column with the right device row. Carriers often wrap \
to different numbers of lines for the same conceptual row, so column position in \
the image matters more than any text ordering.

The text below is a secondary, position-imperfect reference from the same page — \
use it to confirm exact dollar figures and wording, not to determine which \
carrier an offer belongs to; trust the image for that.

IMPORTANT — two ways a single line item can actually represent multiple offers, \
handle both by emitting separate records rather than combining values:
1. One offer applies to several devices at once (e.g. bullets under "All Apple \
Flagships" spanning iPhone 17, iPhone Air, iPhone 17 Pro, iPhone 17 Pro Max) — \
emit ONE RECORD PER DEVICE, repeating the shared values, and set device_group to \
the full comma-separated list of devices that share it (including the device \
itself). For a standalone offer, device_group should just equal item_name.
2. One device has different prices per rate plan tier (e.g. "$0/mo. Good | \
$5.55/mo. Better | $11.11/mo. Best") — emit ONE RECORD PER TIER, with plan_tier \
set to that tier's name (e.g. "Good") and offer_amount set to only that tier's \
price. Do NOT combine multiple tier prices into one offer_amount string. If \
pricing doesn't vary by tier, leave plan_tier blank.

Known carriers from prior reports — use these exact names when the offer is \
from one of them, for naming consistency across weeks: {known_carriers}

For each carrier + device (+ tier, if applicable) combination with a stated \
offer, extract one record: carrier, item_name (exactly one device name, e.g. \
"iPhone 17"), device_group, plan_tier (if applicable, else blank), offer_type \
(your best label: discount, bill credit, trade-in credit, BOGO, etc.), \
offer_amount (dollar figure for exactly this one record, as it appears), \
offer_condition (the requirement text, e.g. "w/ AAL & Any UNL plan"). Skip \
entries with no stated offer ("No offer", "--", "N/A").
{enrichment_fragment}
REFERENCE TEXT:
{page_text}"""

DEVICE_OFFERS_DEDUPE_KEYS = ["carrier", "item_name", "plan_tier", "offer_amount"]


def extract_device_offers(page_number, page_type, page_text, image_bytes, report_date, run_id, known_carriers):
    prompt = GRID_OFFERS_PROMPT.format(page_text=page_text, known_carriers=", ".join(known_carriers),
                                        enrichment_fragment=OFFER_ENRICHMENT_PROMPT_FRAGMENT)
    result = call_gemini_json_with_image(prompt, GRID_OFFERS_SCHEMA, image_bytes)
    deduped_offers = dedupe_by_keys(result.get("offers", []), DEVICE_OFFERS_DEDUPE_KEYS)

    rows, review_rows = [], []
    for o in deduped_offers:
        carrier = normalize_carrier(o.get("carrier"), known_carriers)
        # Only offer_amount is guarded, not item_name — multi-device phrasing
        # legitimately won't appear verbatim in source text even when correct.
        amount_ok = values_present_in_text(o, ["offer_amount"], page_text)
        offer_type = o.get("offer_type")  # schema-enum-constrained — no post-hoc
                                            # normalization needed
        rows.append({
            "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
            "device_category": DEVICE_CATEGORY_MAP.get(page_type, page_type), "carrier": carrier,
            "item_name": o.get("item_name"), "device_group": o.get("device_group"), "plan_tier": o.get("plan_tier"),
            "offer_type": offer_type, "offer_type_raw": o.get("offer_type_raw"), "offer_amount": o.get("offer_amount"),
            "offer_amount_numeric": o.get("offer_amount_numeric"),
            "requires_trade_in": o.get("requires_trade_in"), "requires_port": o.get("requires_port"),
            "is_recurring_credit": o.get("is_recurring_credit"),
            "credit_monthly_amount": o.get("credit_monthly_amount"),
            "credit_duration_months": o.get("credit_duration_months"),
            "offer_condition": o.get("offer_condition"), "values_agree": amount_ok,
            "llm_extraction_raw": json.dumps(result), "extracted_at": datetime.utcnow().isoformat(),
        })
        if offer_type == "other":
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_postpaidDeviceOffers", "flag_reason": "new_offer_type",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": f"offer_type_raw: {o.get('offer_type_raw')!r} — didn't fit any known category.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
        if carrier and carrier not in known_carriers:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_postpaidDeviceOffers", "flag_reason": "new_carrier",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": f"'{carrier}' not in known carrier list.", "status": "pending",
                "flagged_at": datetime.utcnow().isoformat(),
            })
        elif not amount_ok:
            review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page_number,
                "source_table": "silver_postpaidDeviceOffers", "flag_reason": "amount_mismatch",
                "carrier": carrier, "item_name": o.get("item_name"), "offer_amount": o.get("offer_amount"),
                "detail": "offer_amount doesn't appear verbatim in the page's extracted text.",
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })
    return rows, review_rows


def run_device_offers_extraction(run_id, report_date):
    print(f"[{run_id}] Device offers (text+image) extraction starting...")
    auto_classify_unclassified_pages(run_id, report_date)

    known_carriers = get_known_carriers()
    print(f"  Grounding with {len(known_carriers)} known carrier(s): {known_carriers}")

    pages = fetch_bronze_pages_with_image(run_id, list(DEVICE_CATEGORY_MAP.keys()))
    if not pages:
        print("  No matching bronze pages found for this run_id — check show_bronze_state().")
        return

    all_rows, all_review_rows = [], []
    for page in pages:
        print(f"\n--- Page {page['page_number']} ({page['page_type']}) ---")
        if not page["page_image_png"]:
            print("  SKIPPED: no image stored for this page (check PAGE_TYPE_REGISTRY store_image for this type).")
            continue
        try:
            rows, review = extract_device_offers(page["page_number"], page["page_type"], page["text_extraction_raw"], page["page_image_png"], report_date, run_id, known_carriers)
            print(f"  extracted {len(rows)} offer records ({len(review)} flagged for review)")
            print_extraction_results(rows, review, page["text_extraction_raw"])
            all_rows.extend(rows); all_review_rows.extend(review)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            all_review_rows.append({
                "report_date": report_date.isoformat(), "run_id": run_id, "source_page": page["page_number"],
                "source_table": "silver_postpaidDeviceOffers", "flag_reason": "extraction_failed",
                "carrier": None, "item_name": None, "offer_amount": None, "detail": traceback.format_exc()[-1500:],  # full traceback, tail-truncated — not just the message
                "status": "pending", "flagged_at": datetime.utcnow().isoformat(),
            })

    if all_rows:
        delete_silver_rows_for_date(SILVER_DEVICE_OFFERS_TABLE, report_date)
        write_rows(SILVER_DEVICE_OFFERS_TABLE, all_rows)
        print(f"\n[{run_id}] Wrote {len(all_rows)} rows to silver postpaidDeviceOffers (prior rows for this date cleared first).")
        gold_rows = promote_to_gold(GOLD_DEVICE_OFFERS_TABLE, all_rows, report_date, known_carriers)
        print(f"[{run_id}] Promoted {len(gold_rows)} of {len(all_rows)} rows to gold postpaidDeviceOffers.")
    if all_review_rows:
        write_review_rows(all_review_rows)
        print(f"[{run_id}] Flagged {len(all_review_rows)} row(s) for review.")
        print_flag_breakdown(all_review_rows)
        warn_if_review_queue_too_large(all_review_rows)


# ============================================================================
# %% [Cell 18: orchestration — run everything]
# 8. ORCHESTRATION — one call to run every Stage 2 extractor
# ============================================================================

def run_all_stage2_extractions(run_id, report_date):
    """Runs every Stage 2 pipeline in one call, in a sensible order (cheap
    text-only tiers first, image tier last since it's the slowest/priciest).
    Each stage prints its own detailed log; this just adds separators so a
    single pasted transcript is easy to navigate."""
    stages = [
        ("TEXT-ONLY: ci_headlines + national_retail_readout", run_text_only_extraction),
        ("TEXT-ONLY: major_offer_changes", run_major_changes_extraction),
        ("IMAGE: voice_plan_comparison", run_voice_plan_extraction),
        ("IMAGE: business_internet_comparison", run_biz_internet_extraction),
        ("TEXT-ONLY: prepaid_price_grid_detail", run_prepaid_grid_extraction),
        ("TEXT-ONLY: postpaid_service_offers", run_service_offers_extraction),
        ("TEXT-FIRST GRID: bts / cable_mvno / prepaid_brand", run_carrier_grid_extraction),
        ("IMAGE: device offers incl. business", run_device_offers_extraction),
    ]
    for label, fn in stages:
        print(f"\n{'#' * 80}\n# {label}\n{'#' * 80}")
        fn(run_id, report_date)

    # Per-stage warnings above catch one table generating an outsized flag
    # count on its own; this catches the case where several stages each stay
    # under threshold individually but sum to a real backlog across the run.
    # Queried directly from the review table (not summed in-memory across
    # stages) so it reflects what's actually sitting there for this run_id.
    total = list(bq.query(
        f"SELECT COUNT(*) AS n FROM `{REVIEW_TABLE}` WHERE run_id = @run_id",
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]),
    ).result())[0]["n"]
    print(f"\n{'=' * 80}\nTotal rows in review for this run: {total}\n{'=' * 80}")
    if total > 100:
        print(f"That's over the 100-row practical ceiling for manual review, even though no single "
              f"stage may have tripped its own warning above. Worth checking the per-source_table "
              f"breakdown in {REVIEW_TABLE} directly before starting notebook 2's review pass.")

# run_all_stage2_extractions(run_id="...", report_date=date(2026, 6, 19))